# Proyecto final — Red de transferencias de Ethereum

**Estructura de Datos — Universidad Industrial de Santander**

**Equipo:** Jhon Arciniegas, Pablo Álvarez y Emerson Ballesteros

---

Este notebook es **autocontenido**: incluye los datos de la muestra y el
código de los cuatro algoritmos en sus propias celdas. No necesita clonar el
repositorio ni instalar módulos del proyecto.

| Tarea | Pregunta | Algoritmo | Responsable |
|---|---|---|---|
| 1 | ¿Cuál es el camino más barato? | Bellman-Ford | Pablo Álvarez |
| 2 | ¿Qué nodos son más importantes? | Grado + PageRank | Jhon Arciniegas |
| 3 | ¿Qué grupos de nodos hay? | Louvain | Emerson Ballesteros |
| 4 | ¿Cuál es la red mínima? | Kruskal + Union-Find | Grupal |

> **Cómo ejecutar:** en Colab, ejecuta las celdas en orden. La celda de datos
> ya trae una muestra real de USDC. La última sección usa el Parquet completo
> si está disponible en Drive.

## 0. Capa de datos: el grafo

Esta celda define la estructura compartida: un grafo dirigido y ponderado.

- **Nodos**: direcciones Ethereum.
- **Aristas**: transferencias `(from_address, to_address, value)`.
- **Peso**: cantidad transferida.

El resto de celdas usan el mismo objeto `grafo`.

In [ ]:
"""
grafo.py
========
Carga del dataset y construcción del grafo.

En este proyecto modelamos las transferencias de un token ERC-20 de Ethereum
como un grafo DIRIGIDO y PONDERADO G = (V, E):

    - V (vértices): las direcciones (wallets, exchanges, contratos).
    - E (aristas): las transferencias de una dirección a otra.
    - w(e) (peso): la cantidad transferida.

La carga acepta **CSV** y **Parquet**. Para Parquet se usa DuckDB solo para
leer y muestrear el archivo; los algoritmos del proyecto siguen siendo Python
puro y reciben el grafo ya construido.

Solo usamos estructuras nativas de Python: diccionarios y listas.
"""

from collections import defaultdict, deque
from typing import Dict, List, Optional, Set, Tuple

# Lista de adyacencia: cada nodo apunta a una lista de (vecino, peso).
AdjList = Dict[str, List[Tuple[str, float]]]

# Nombres de columna que aceptamos para cada dato. Se comparan sin distinguir
# mayúsculas/minúsculas, así funciona con distintos exports del dataset.
COLUMNAS = {
    "from": ["from_address", "from", "sender", "from_addr", "fromaddress"],
    "to": ["to_address", "to", "receiver", "to_addr", "toaddress"],
    "value": ["value", "amount", "token_amount", "quantity", "val"],
    "token": ["token_address", "contract_address", "token", "tokenaddress"],
    "time": ["block_timestamp", "timestamp", "block_time", "date", "time"],
    "block": ["block_number", "block"],
}

DIRECCION_CERO = "0x0000000000000000000000000000000000000000"
DIRECCIONES_CERO = {DIRECCION_CERO, "0x0", "0x00"}


def _elegir_columnas(disponibles: list[str]) -> Dict[str, Optional[str]]:
    """Mapea cada dato lógico (from, to, value...) a su columna real.

    `disponibles` debe venir en minúsculas y en el orden del archivo; el
    resultado conserva el nombre real de la columna para poder citarlo en SQL.
    """
    return {
        clave: next((opcion for opcion in opciones if opcion in disponibles), None)
        for clave, opciones in COLUMNAS.items()
    }


def _sql_texto(valor: str) -> str:
    """Escapa un texto para usarlo dentro de una consulta SQL."""
    return "'" + str(valor).replace("'", "''") + "'"


def _sql_columna(columna: str) -> str:
    """Cita un identificador de columna para SQL."""
    return '"' + str(columna).replace('"', '""') + '"'


class Grafo:
    """Grafo dirigido y ponderado representado con una lista de adyacencia."""

    def __init__(self) -> None:
        # {origen: [(destino, peso), ...]}
        self.adj: AdjList = defaultdict(list)
        # Todos los vértices del grafo.
        self.nodos: Set[str] = set()
        # Lista de aristas como (origen, destino, peso).
        self.aristas: List[Tuple[str, str, float]] = []
        # {dirección: {"nombre": ..., "tipo": ...}} para mostrar resultados legibles.
        self.etiquetas: Dict[str, Dict[str, str]] = {}

    # ------------------------------------------------------------------
    # Carga de datos
    # ------------------------------------------------------------------

    def _agregar_arista(
        self,
        origen: str,
        destino: str,
        peso: float,
        limpiar: bool = True,
    ) -> bool:
        """Agrega una arista y devuelve si fue aceptada.

        Con `limpiar=True` se descartan transferencias inválidas para el
        análisis: dirección cero, auto-transferencias y valores no positivos.
        """
        origen = str(origen).strip().lower()
        destino = str(destino).strip().lower()
        peso = float(peso)

        if limpiar:
            if not origen or not destino:
                return False
            if origen == destino:
                return False
            if origen in DIRECCIONES_CERO or destino in DIRECCIONES_CERO:
                return False
            if peso <= 0:
                return False

        self.nodos.add(origen)
        self.nodos.add(destino)
        self.adj[origen].append((destino, peso))
        self.aristas.append((origen, destino, peso))
        return True

    def cargar_transferencias(
        self,
        ruta_csv: str,
        max_filas: Optional[int] = None,
        limpiar: bool = False,
    ) -> "Grafo":
        """Lee un CSV de transferencias y llena la lista de adyacencia.

        El CSV debe tener (al menos) columnas equivalentes a:
            from_address, to_address, value
        (se aceptan alias como `from`, `sender`, `to`, `receiver`, etc.).

        Parámetros:
            ruta_csv: ruta al archivo CSV.
            max_filas: opcional. Leer solo las primeras `max_filas` filas
                       (útil para probar con una muestra).
            limpiar: descartar dirección cero, auto-loops y valores <= 0.

        Retorna:
            self, para poder encadenar llamadas.
        """
        import csv

        with open(ruta_csv, newline="", encoding="utf-8") as archivo:
            lector = csv.DictReader(archivo)
            campos = lector.fieldnames or []
            disponibles = [c.lower() for c in campos]
            reales = {c.lower(): c for c in campos}
            mapa = _elegir_columnas(disponibles)

            faltan = [k for k in ("from", "to", "value") if not mapa[k]]
            if faltan:
                raise ValueError(
                    "No encontré las columnas necesarias: "
                    + ", ".join(faltan)
                    + ". Columnas disponibles: "
                    + ", ".join(campos)
                )

            col_from = reales[mapa["from"]]
            col_to = reales[mapa["to"]]
            col_value = reales[mapa["value"]]

            for i, fila in enumerate(lector):
                if max_filas is not None and i >= max_filas:
                    break
                self._agregar_arista(
                    fila[col_from],
                    fila[col_to],
                    fila[col_value],
                    limpiar=limpiar,
                )

        return self

    def cargar_parquet(
        self,
        ruta: str,
        max_filas: Optional[int] = None,
        muestra: Optional[int] = None,
        token: Optional[str] = None,
        desde: Optional[str] = None,
        hasta: Optional[str] = None,
        semilla: int = 42,
        limpiar: bool = False,
    ) -> "Grafo":
        """Lee un Parquet (o una muestra) usando DuckDB.

        DuckDB solo se encarga de leer, filtrar y muestrear; el grafo se
        construye igual que con el CSV, usando Python puro.

        Parámetros:
            ruta: ruta al archivo Parquet.
            max_filas: leer solo las primeras N filas.
            muestra: tomar una muestra aleatoria de N filas (recomendado para
                     el dataset completo).
            token: filtrar por `token_address`.
            desde / hasta: filtrar por `block_timestamp` (texto ISO) si existe.
            semilla: semilla de la muestra aleatoria.
            limpiar: descartar dirección cero, auto-loops y valores <= 0.

        Retorna:
            self, para poder encadenar llamadas.
        """
        try:
            import duckdb
        except ImportError as exc:  # pragma: no cover - depende del entorno
            raise SystemExit(
                "Falta duckdb para leer Parquet. Instálalo con:\n"
                "    pip install -r requirements-datos.txt"
            ) from exc

        con = duckdb.connect()
        try:
            campos = list(con.read_parquet(ruta).columns)
            disponibles = [c.lower() for c in campos]
            reales = {c.lower(): c for c in campos}
            mapa = _elegir_columnas(disponibles)

            faltan = [k for k in ("from", "to", "value") if not mapa[k]]
            if faltan:
                raise ValueError(
                    "No encontré las columnas necesarias: "
                    + ", ".join(faltan)
                    + ". Columnas disponibles: "
                    + ", ".join(campos)
                )

            select = (
                f"CAST({_sql_columna(reales[mapa['from']])} AS VARCHAR) AS from_address, "
                f"CAST({_sql_columna(reales[mapa['to']])} AS VARCHAR) AS to_address, "
                f"CAST({_sql_columna(reales[mapa['value']])} AS DOUBLE) AS value"
            )

            filtros: list[str] = []
            if token:
                if not mapa["token"]:
                    raise ValueError("Pediste `token` pero el Parquet no tiene columna de token.")
                filtros.append(
                    f"CAST({_sql_columna(reales[mapa['token']])} AS VARCHAR) = "
                    f"{_sql_texto(token)}"
                )
            if desde or hasta:
                if not mapa["time"]:
                    raise ValueError(
                        "Pediste una ventana de tiempo pero el Parquet no tiene "
                        "columna de tiempo (block_timestamp)."
                    )
                col_tiempo = _sql_columna(reales[mapa["time"]])
                if desde:
                    filtros.append(f"CAST({col_tiempo} AS VARCHAR) >= {_sql_texto(desde)}")
                if hasta:
                    filtros.append(f"CAST({col_tiempo} AS VARCHAR) <= {_sql_texto(hasta)}")

            consulta = f"SELECT {select} FROM read_parquet({_sql_texto(ruta)})"
            if filtros:
                consulta += " WHERE " + " AND ".join(filtros)

            if muestra is not None:
                n = int(muestra)
                consulta += f" USING SAMPLE reservoir({n} ROWS)"
                if semilla is not None:
                    consulta += f" REPEATABLE ({int(semilla)})"
            elif max_filas is not None:
                consulta += f" LIMIT {int(max_filas)}"

            try:
                filas = con.execute(consulta).fetchall()
            except Exception as exc:  # DuckDB antiguo sin REPEATABLE/reservoir
                mensaje = str(exc).lower()
                if muestra is not None and ("parser" in mensaje or "syntax" in mensaje):
                    base = consulta.split(" USING SAMPLE ")[0]
                    filas = con.execute(f"{base} LIMIT {int(muestra)}").fetchall()
                else:
                    raise
        finally:
            con.close()

        for origen, destino, peso in filas:
            self._agregar_arista(origen, destino, peso, limpiar=limpiar)

        return self

    def cargar(self, ruta: str, max_filas: Optional[int] = None, **opciones) -> "Grafo":
        """Carga transferencias detectando el formato por la extensión.

        - `.csv`  -> `cargar_transferencias`
        - `.parquet` / `.pq` -> `cargar_parquet`

        Las opciones específicas de Parquet (`muestra`, `token`, `desde`,
        `hasta`, `semilla`, `limpiar`) se pasan tal cual.
        """
        if str(ruta).lower().endswith((".parquet", ".pq")):
            return self.cargar_parquet(ruta, max_filas=max_filas, **opciones)

        sobrantes = set(opciones) - {"limpiar"}
        if sobrantes:
            raise ValueError(
                f"Las opciones {sorted(sobrantes)} solo aplican a archivos Parquet."
            )
        return self.cargar_transferencias(ruta, max_filas=max_filas, **opciones)

    def cargar_etiquetas(self, ruta_csv: str) -> "Grafo":
        """Lee un CSV con el nombre legible de cada dirección.

        El CSV debe tener las columnas: address, name, type.
        """
        import csv

        # Algunos exports de etiquetas vienen en latin-1 (nombres con tildes).
        filas = None
        for codificacion in ("utf-8", "latin-1"):
            try:
                with open(ruta_csv, newline="", encoding=codificacion) as archivo:
                    filas = list(csv.DictReader(archivo))
                break
            except UnicodeDecodeError:
                continue
        if filas is None:  # pragma: no cover - archivo binario inválido
            raise ValueError(f"No se pudo leer el CSV de etiquetas: {ruta_csv}")

        for fila in filas:
            addr = fila["address"].strip().lower()
            self.etiquetas[addr] = {
                "nombre": fila.get("name", "").strip(),
                "tipo": fila.get("type", "").strip(),
            }
        return self

    # ------------------------------------------------------------------
    # Utilidades
    # ------------------------------------------------------------------

    def conservar_componente_mayor(self) -> "Grafo":
        """Deja solo la componente conexa (no dirigida) más grande.

        Un muestreo aleatorio deja muchos nodos aislados. Quedarse con la
        componente más grande produce una red conectada y resultados
        interpretables, sin volver a leer el archivo.

        Retorna:
            self, para poder encadenar llamadas.
        """
        adyacencia: dict[str, set[str]] = defaultdict(set)
        for origen, destino, _ in self.aristas:
            adyacencia[origen].add(destino)
            adyacencia[destino].add(origen)

        visitados: Set[str] = set()
        mejor: Set[str] = set()
        for inicio in adyacencia:
            if inicio in visitados:
                continue
            componente: Set[str] = set()
            cola = deque([inicio])
            visitados.add(inicio)
            while cola:
                nodo = cola.popleft()
                componente.add(nodo)
                for vecino in adyacencia[nodo]:
                    if vecino not in visitados:
                        visitados.add(vecino)
                        cola.append(vecino)
            if len(componente) > len(mejor):
                mejor = componente

        if not mejor or len(mejor) == len(self.nodos):
            return self

        aristas = [
            (a, b, w) for a, b, w in self.aristas if a in mejor and b in mejor
        ]
        self.adj = defaultdict(list)
        self.nodos = set()
        self.aristas = []
        for origen, destino, peso in aristas:
            self._agregar_arista(origen, destino, peso, limpiar=False)
        return self

    def nombre(self, addr: str) -> str:
        """Nombre legible de un nodo, o la propia dirección si no está etiquetada."""
        return self.etiquetas.get(addr, {}).get("nombre", addr)

    def resumen(self) -> Dict[str, float]:
        """Estadísticas básicas: número de nodos, aristas, densidad y peso."""
        n = len(self.nodos)
        m = len(self.aristas)
        # Densidad de un grafo dirigido: |E| / (|V| * (|V| - 1))
        densidad = m / (n * (n - 1)) if n > 1 else 0.0
        peso_total = sum(w for _, _, w in self.aristas)
        return {
            "nodos": n,
            "aristas": m,
            "densidad": round(densidad, 4),
            "peso_total": round(peso_total, 2),
        }


def cargar_grafo(
    ruta: str,
    max_filas: Optional[int] = None,
    etiquetas: Optional[str] = None,
    limpiar: bool = True,
    **opciones,
) -> Grafo:
    """Carga el dataset UNA vez y devuelve el objeto compartido `Grafo`.

    Es la función que usa el notebook. Las cuatro tareas reciben
    `grafo.aristas` y construyen su propia estructura; no vuelven a leer el
    archivo.

    Por defecto limpia transferencias inválidas (dirección cero, autolazos y
    valores <= 0). Usa `limpiar=False` si quieres cargar el archivo tal cual.

    Ejemplo:
        grafo = cargar_grafo("data/sample/transferencias.csv")
    """
    opciones.setdefault("limpiar", limpiar)
    grafo = Grafo().cargar(ruta, max_filas=max_filas, **opciones)
    if etiquetas:
        grafo.cargar_etiquetas(etiquetas)
    return grafo

In [ ]:
# ============ DATOS DE LA MUESTRA (embebidos) ============
# La muestra está comprimida dentro del notebook para que sea autocontenido.
import base64
import csv
import gzip
import io
import math
import random

SAMPLE_B64 = """
...
"""

ETIQUETAS_CSV = """
address,name,type
0x0681d8db095565fe8a346fa0277bffde9c0edbbf,Binance 4,cex
0x1062a747393198f70f71ec65a582423dba7e5ab3,Huobi 9,cex
0x11111254369792b2ca5d084ab5eea397ca8fa48b,1inch.exchange,dex
0x2faf487a4414fe77e2327f0bf4ae2a264a776ad2,FTX Exchange,cex
0x3f5ce5fbfe3e9af3971dd833d26ba9b5c936f0be,Binance 1,cex
0x564286362092d8e7936f0549571a803b203aaced,Binance 3,cex
0x59a5208b32e627891c389ebafc644145224006e8,HitBTC 2,cex
0x5c985e89dde482efe97ea9f1950ad149eb73829b,Huobi 5,cex
0x6cc5f688a315f3dc28a7781717a9a798a59fda7b,OKEx 1,cex
0x6f803466bcd17f44fa18975bf7c509ba64bf3825,Poloniex: USDC,cex
0x794e6e91555438afc3ccf1c5076a74f42133d08d,OasisDEX,dex
0x94fe3ad91dacba8ec4b82f56ff7c122181f1535d,Faa.st,cex
0x986a2fca9eda0e06fbf7839b89bfc006ee2a23dd,Bitmax 3,cex
0xa12431d0b9db640034b0cdfceef9cce161e62be4,HitBTC 3,cex
0xa910f92acdaf488fa6ef02174fb86208ad7722ba,Poloniex 4,cex
0xd0808da05cc71a9f308d330bc9c5c81bbc26fc59,Coinswitch,cex
0xd2045edc40199019e221d71c0913343f7908d0d5,Paradex,dex
0xd551234ae421e3bcba99a0da6d736074f22192ff,Binance 2,cex
0xf92c1ad75005e6436b4ee84e88cb23ed8a290988,ParaSwap,dex
0xfa4b5be3f2f84f56703c42eb22142744e95a2c58,Huobi 11,cex
"""

# Decodificar la muestra
texto_csv = gzip.decompress(base64.b64decode(SAMPLE_B64)).decode("utf-8")
aristas = []
for fila in csv.DictReader(io.StringIO(texto_csv)):
    aristas.append((
        fila["from_address"].strip().lower(),
        fila["to_address"].strip().lower(),
        float(fila["value"]),
    ))

# Etiquetas
etiquetas = {}
for fila in csv.DictReader(io.StringIO(ETIQUETAS_CSV.strip())):
    etiquetas[fila["address"].strip().lower()] = {
        "nombre": fila["name"].strip(),
        "tipo": fila["type"].strip(),
    }

# Construir el grafo compartido
grafo = Grafo()
for origen, destino, peso in aristas:
    grafo._agregar_arista(origen, destino, peso)
grafo.etiquetas = etiquetas

print("Muestra cargada:")
print("  nodos :", len(grafo.nodos))
print("  aristas:", len(grafo.aristas))
print("  etiquetas:", len(grafo.etiquetas))
print("  resumen:", grafo.resumen())

## Sección 1 — Descripción y preparación del dataset

**Responsable:** Pablo Álvarez

El dataset es **Ethereum Exchanges (AlphaCore)** de Stanford SNAP:

| Dato | Valor |
|---|---|
| Contenido | 28 redes de tokens ERC-20 (una por token) |
| Transferencias | ~38,9 millones |
| Tamaño CSV | ~8,3 GB |
| Periodo | 16-oct-2018 a 04-may-2020 |
| Etiquetas | 296 exchanges: 247 `cex` y 49 `dex` |

Cada fila es un evento `Transfer(from, to, value)` de un contrato ERC-20.
`value` está en la **unidad mínima de cada token**, así que no se pueden sumar
montos de tokens distintos.

### Decisiones de modelado

1. **Un grafo por token.** Mezclar USDC (6 decimales) con DAI (18) no tiene
   sentido.
2. **Limpieza igual a la del paper AlphaCore:**
   - quitar valores ≤ 0;
   - quitar “polvo”: valor < valor máximo del token / \(10^{10}\);
   - quitar autolazos;
   - conservar la dirección cero (mint/burn) y reportarla aparte.
3. **Agregar transferencias repetidas** del mismo par en una sola arista:
   `value` = suma, `n_transferencias` = cuántas fueron.

Aquí se muestra con **SoftChainCoin**, una red real pequeña que usan los
autores del dataset. Las tareas 1–4 usan la muestra de USDC porque tiene más
exchanges y comunidades.

In [ ]:
# ============ SECCIÓN 1: DESCRIPCIÓN Y LIMPIEZA DEL DATASET ============
import base64
import csv
import gzip
import io
import math
from collections import defaultdict, deque

import matplotlib.pyplot as plt

RAW_B64 = """
H4sIAAAAAAAC/72d+440u23t/8+rpNDQhdTlaQJdkSCJN2A7B37882PNdxtPn6D6VO/PsLG9Z3qKJYlcXIui1H//4z/XX/6tzfnX
9be/Hfuvf/z3j3/5+x8//u//af/1P+vo//XH+M9/+8v//Hdff/2Xv/2x/z7+vf3HX8Yf//GXw/1Ds6/L91FKWbs5N4f2MGsNo8Uk
pVQfaiup8tEeJaacS9atcyVfZ597S9bIz3zyzs/RY5tHXP/qypFCyFr9HYu15iFlLN+2GyskLTMuGa6vFl1zq9dRSyzhiI+EzXza
TD7csVk0xub6rNFjIMWVxyjLe5+GLy3OlLuvObXDq7P/YDPGUOW32Mz6i8lyx+Sqwzfx3fUW8iqu+1iGWz6b7VR8D8nVOvYhTk+D
osl9MXj5Kaxllt5SW2X2MiVkBtar7B4XHx9Oxohlh7JtXt9hL8SWxRw25pZ2L152235jJ7UefAor7epie5s9LTFLiW3FUcVlv3r0
Xofk0Lf3McbRu/j0LnujuxW2lzbSWgxmyey5qHbtyXeddUa/Vhnvshf9iFPULc2yeslxxLyY3tSk1a6eR0RG299lD2hJK64usU98
phLvYY+RZU+3Ss09hujSmu+y16rXprP7uVuYYW0iLvSZldgT3cWFtmqa+V32dkm2RnySsFCWCn+c27uND40obY4QQ6pvi4euua65
s3eluDbG3CuG4mMdDHpNKXXnvX14m7/UPOMQ1s1vlnFFza41QkCbDzKCnykGwOhd9qoT4mFaADAYnDSvunruazCquGcYDj/Kb5vP
kfL2PY2aWsEt8coyZO2iJKeRA0/QsMqSd9mTNYKm4ZSkWLV3/GWMHYf306VUXYmk4Bzq29ZvlBBqB8acZB91KpmAqZypAKBRyi4L
xHtbvI+QpfWU+Id679KIe+E+frfhclh+z7irdP+2+JOWifS+QxpM5h7KH2Bi1epnDJLjFp+Le5e91JrG2iufFCErrxBVQwM5p3j7
adXMANfb5nM1cRN6Ji4xm2u11Kc6Pl2lzGa+2iM07m14nVsIln16KswtPKwXGZJkR5eFGV1z5e7ehi/BskEj5sl3GY7jvJuQiQLZ
mSmp1GUZsr8NrwtvPyOMis/XTErQlpe2za9j6znnlHpN71s/ja2I31vXhk0EN6rLY8osqcTIyBJ8KvCbt42v97aar801rRuqzVgY
otpPVCfJcbadw9vWT2saiX9NhDlMMGnys7TcMpHRY466Vu2lv20+JcHGiidFRChxWVFqA+MGdvPcA78JDFLj+/L7Ts2NWbpLMzR1
u7fN83uJKfZEgip1+ijvw7NEZo81Rbh0akiGsdtKRB5Jl7AwHRF6/9/x+rLgAl+aNnLOWJJb5E189tplVR8DqRGt0fbwfTM+lJma
Rda4/hbJAjlV/3AxYRUMd3orDxbdREIf3djLAmVI/GOLMkBctE7YoiBcDpXy8O+wWJsRFT96QcrimJiOaOwcEYJzB6+sq/NpH1As
fSSJ+g6rmRzUnSutbZmzJ9dDy6SLZHwqLPjUdgUxc8AFTgl626JEX4A3WPfYHQ/QsDe+46YjShpaGB6XRx5HSDCQh3/L5HYX69wl
V7I7j4+GQmvXHAMkA3jtgiuCtYRJ1rcMM6iRpmbCHazrO6cxIatursVsj+wqZMqdidGnhxP/luWc8N7lwTkSlaat/NE2eYi7DnEb
tboY/d7HmxZzzFYlxDz2cBhGcYMzxYmxxoywqDsivfc4smp+yDtM7hHrHlM7s9vrhA3gqbhs2S5kH2ZB40Du5EB5vGWMy/GzLjAN
IWn55goMAFHFH8lyES6wpq4JRa3vMYhQy6At4+Bj0e81pzHFsZxA3cKKUQJpKx6B7PaQ8JYI4UNwRF9yq3NpqFabQRZDH6Hi/IFP
pTDRJEqIyCOGt0CBonUJvTgl9RQ8+lRgjKjGHp0WKwMUWEI+NMXy/zT4Qi65nC3Dt+yFRYl3LI5eISCapmsVMTyg+j3VxaAkMLEJ
4LWqDXznw3fEh3wnP4PeJA0hdeAiQkDOBXyPNUNYSPMOMhAnSrZkhGemFMjeLUZQWWv0EwyrZRaoZPiB96Dprt1qRxU9lWs6wiN8
q9aKFP+1cnr5QQZziI4NwASSdCMrkpK3qRxy5U47dAmqyPEjfR+lBv91lJcny3R4q7hLGqXrhohP52eVnKeW7KCyZA1tK+Sf84rF
9HWMV/ES5bhLCkSeAN+j5JCbC7MtEom9H+6VojSR+Qu0Moyv4XE531rxdC4GnWqXlGNTm8npPYTV6mPSrP7WwNZPqVlD+lp5v5z9
YCDbJmQt3ma1DND05vYuoUA0XSo5C5EDEfmcKDXq1zi5DJyglIoQ9LMO5CpUjuGOYXVG1hjqh4xlihQqYBj7YTB/BYLLqQHoadKT
BzDwYdy1mLPDoisCLMw5WltwlLH2jyyCS3z1n8t0zUJYMrGBjhNeIe4hAcWTAagUWVCF7ziWkbh03w0m/S0GvdfvFvPX3YzLPNj9
AwcqvcgkNoKPCGV8FpO4qnfREwzghpVYTsp8GtTyxFevej0fzbLjQOpUpz7z0BWglSMw6BB39Dr80j3ypwBBgX6VXJczkQEBwwMM
qlvMeHG1awRputWNfFYZG8qDrobTfYg85vuJ5LpMgclvzX4dDHmzFR5RdAFNR8jirjuB/lYqR/x8S1rkvPh77Gn6NkJ9QgQuY/QL
C/kDzpPGr756mYuCTbXIStHIT4qsjw/AfpGeByjXgQkx6AeGv9NW1PTXIV5OfC8M8UeORD9/RZzLaegFiz8zVinhq8nL2PxKQH6D
ccD1q6texuYXDH6H8RqeEMjLwPWCwe8YV8uTvHEZm18Z4feKAA+XJ35zEbVesPgd4LD7xOLlCHvF4kcwYrg+kQFXYeuVSf1Wf9BQ
5OsyXhZp+DTPbJDKzhslmFSZbSLN9qpwTHw9Bj99K/JNz5lJFvJrorpMz14a5E8mp+rL14FefvtXrH4faHLhSYj8GRnZf2/cSP4Z
ml9+dTAxFAOoAWKo4gVta0xz5RVa8rrwcsh2yvXQb8oVWH7CkP+MqPw5SPJHvDPImCJ0gQcRta0gAlOSiNc1xGYD8jMein5ZR0z6
YbKGUG6ZlBZ9s3r2dhv+7hoyvqKPYnOotLZ5hdzADRJIdvkR/Qmy1UDmTlng8jp8L0RU5G++NbfepFiFTEcUUFXiaJIN6wo7h+ad
1rL6sIHGE4Aq8x1+i//8GCJocitG+HVHjAJWYY+GE7F8za/V+oBNESVkuwQZPUItjPIhH2tZ4pPWsRfMOmTmBItmV3R9834BiVvD
XC1NSCfANWOq8Yg5PdTQG7MRzVjrHbOb9EL8e3ys1BihG/y3bMSSteihvqtjCaXjuRV7npx4y55tEoY10ggwjdgZc4az8DEgs4fQ
HaxsLL+xFf0DimdGkYbpjlEcNBX4noXhyh5q5GUnxleXlBl1IikQH3II2f4R8jm1ZDJ3a0Xn1kQS2MwvqpD1i00qr7DJOZPEXkO3
WqV+VNEBWl9uTe2U0jDVV947u74SH5u5RUAE3Sub+Ir8+yaFpfQACswoPEF+CwR9yyjQmt+VUb5bZHrjLUSQsTSRzacwt8A604Bc
G957iVbkC7ErvqSHz/HhLUxSTiH8lhKz/5BdZLxn1ODPYCPl27RmK93+loX8brFAn/2dgu9laf9NzEa43pOY/DMqBD8tVrmFApX5
Jfp9W27uoDhalTCYAbzMXExCKTioHKoM7lGiGRURfwvVR+tuzxGjY7Z3QXrPNAcY65gRUhrwoyHYto/RAoN+5+4YzK2hyOBYXQgc
wfryzmh+1l2x3lqGanX9YZBB3wJzuH5UXd3XwXJXJfVb5XvqcKtJ0Yjjpd4mxMfHmB+nLBGH5rhl1kvT5FOb6v0uZQ4gvdUs6jpq
hqjC69FyBeixHgn3cOQtDHvVe8nr8qZG0Ay64LoSfPZfAeGy5CCBVRklyFqzxrBDYsatpc4lqduLgieyYpd2eGfmiovpjrk4hrMe
qYaPsGbF6kOFNa4jaUYaxsXEEio/zEm850EwxNwFIgAB2Tn7AKLNMPa0XcpG8OhqAbKegi9YhJGEW2R9F0aEX1ijkoMalBJ229VH
lnbPkNX3gs/MAzvpw+JN7rya+t0H41g9VpCxp9DbSIIKF9izkkSnG/EQg2QzqaryZPf36nNMes0CCyYJBx1IRzfUyvsBpp6rLus9
na75+YvJ5PzXeb38nBcm5KdJcNfd8dXW8ZDY9+xogAqmo1+JEW0hjzg7KCszOJHTV6F0qrdCY3sHhILbAB1sdZUN+SEoEHgrpqFD
WgAD+g9zzxT7n7At+gNsNBTy2G/JzL/aBHlvhUeFmc/tIUV5+xJIzqO1aPv4aIMOum8Y29zoEIgbFqNVm+5YHABnXBtN5zuoM7sg
aiXVTAzwyRqczh03CxkY5MNOXmm0quEtITJIRS1DL8nQQXAg2O0MadQIBGU/ozW/Z+RWQWI9xDKWWqn1FiVw2WAVeJu8fwslFWtx
1eZKTM0tc+61fDIli1x/pNMo03AL8GZEYA2j9AJOteXjnCMx6UVTmAy298BqE5dBU34Ec6OY871qU0plwywyxJoAc44FDHjRZKzR
FphZrc7KEtjJD3dOr7hU72nomNb0/IpZdDisX2nIbKTNAjCS0zLj3nHCRErxWAQW3K30lWK0MxB2Miq1MVvRwtCK79F0re9hNdeK
h1SiceQhVivA6s3q4dj4SiUXJCbXOgBcqWsu+OWumxfQ0axL3NuuZURGn1YTeuMe3Zq175TsE9bVkJdzeUqdbXpoHmiuBvQlHECQ
xSiM9kkOuxztIHzdUktl6QQOvWGLfjrH551fC6hXtCicIf0KDDb5t2QCqVvxI5QIw5nQjTo6FB7SExMAgaIu5JjQzn228lBTfckq
ibeAIc7S0vLIXYJz+x3chroz9GInDpiDaPVYrUclT8lDrTCSoET3igYahpZWBnObxyZSRI3tlYSDDubRRjwg7/gRlONhNsWS2heb
l1/fjhlYQ8WKhRS3+wodLaSGxoQ/KTwFgD2ULp9GCsHRW4qT5zffNv9D5Vkzt2+jQqcU9uOgKNN3q31DM2ON8kjltBp4ta+bb1cH
8MILfhpr9E+2/C4PgMWIOohLgK4YHLZg5TV0So9bdqjnaacGX/o8VvFPuMrlR72ye/urVT56K43joJCS0Jl48iT814U1Sh4D+asd
n3RukG5BQuSoPqy3MuUi9SsQXlZaUCTgC5y3LXqACNa7RSYkWoXEKmvE4buOAIQ9/LnHjdTNpKCA8cKAb424B9dyyGsD7XFap1cu
3cpmeKIbdihKeOnsjowojQ+1EVeHwvpKRK8inVUXo7Mt9LZqXbo04exQsJHztl3nCWVShL77FRSzI988aV25+qgXWx5+WLXjdbdc
Ck6wa8t+2IH/Dl/r8EFcX0bbK7DigKaIteqKlVLNKFBZbvEI1s5PYTEi8ZYbonit6KqkzkLbXCH3/NjRcg1AHE+jaJavIXv5Se4f
8LKN0+c54aSaOox+RkQQ+msGIK6QayG8+bNRieEeY+L1dk08GLVjZ1c9gZpDQk4EOzU4VRbResCX8kMho3i73mMuqfZobGOgCos1
oPjB2IPvhbxGQO1OBOwgB8NL9eHNaIzu2ey+UBZDS+RozSa9E6lEFt7TVxThx8uKLDj+zuvwxdeKRSY21zu41EeaBW1pRyFYywoL
jnOsAbnq1YW2rR2ZVQaX7ERtkJqCC+R0jWaeAd0acLazn9BS590Os9kpybxRVz6n3pEZxe2mcDqr6uZHMVKarcB7j/JriLx7chhq
0NI1o0TGDLJAjGtcI7syyj6CTcijZFMacE2J9+TjdjsRXiuXkXi0LTGpMNp9BSTGIiuMFYI1R7Gs+sjuwy5aWX+LXWallIqwyqdd
KxDcsls32mmiakpT2yGsvnfoR4+EJv4F9ShhbqPigbRhVNHuTLm3TTCDndD057FehEGW1RSlPFyZKeHiFd3TTCqw0Nk/8tkWhln4
5i0uTqSUsFMGBFfM0zM8otjlkefgjxEuYt3AR9Ysjw+bMNxb8wsQD7hCk7ktnMFvKMPosPOeGb6E5GzrHbCIITG/3ugTlCO4WzNc
3AAPZifXgS2AQWxAYXHaXS6LJc9WgPXuEKu7YpKU4O8pHZ+YWw/mBttWZLxzQZ7GHrYBoky07f20tU7NQWibTeIm3jm4cbnkpvXb
lTeIH3jJb9nH8x9HRQqZVe41haDUwtgFsE1TeWBq+BGeuTJiO8EApnQIwKHEC7B02kxPqPDlB70ys7/YzE+6UP6Uqigr9N2olpvd
U64125+G+4F+OF7tvVqvUvRGroN6gKrFfOQERj2ITIsWX9O9Sg/r45ddP2PuNkH6FGaqTTKEudoVIDGt4aYeIYcQH8EqhoCCD09g
4eqj7KoYuEPLxcsi0aRpt2CZcGnW2b51b3I7nOqfrYYn23mXH/XKuv5qFXl1i3nb1UytMRFjT2tmLw11ojPYPTXoRFhFtddthx3h
M1cS6626Faf4TAckZPbSIKFINmumJadrQlSal+VIZjvgCiE/Yj1diXXSr/Ts8rMgwaEjC0eHjJHafOjW9aCE+pg7OuvVCpC0of9s
9lnP2OVnvdJB8clsKTnf28aI/D9BxExrKelew5x9VCtCa5wnSdi79fPGMTNI1q2/xeBHP2dR5Pu9RBo36BZgez4s0QZ0Jo9cldis
BlcSHw/erZ8G9R4dW6aZYCl26ma20QkWKF9AVVTbMO4Z8b1dPnG35opiNG2M3RrubSWcY0LrtJy7G+m8gyyv6XuKJIiGlimIOBwX
6AUYLJmql3tlllZq8ZAyU6HDn3djxYVwY02j9IRcTJDhXg4JLiYr9JxjhajewobtW7ViRuukFzh93GWv7dXq7aEudKtK69HO0pMc
H/XclypWR7s1Wr/JyQ0ZGjShoOKCjLleOjNgZx9IJwGUVLtbyub2oxZAYvY53mMsJAbnY204E9oRzAhugmcNGLCydKjAGivr8QEr
8J+jzd7Vcq+NlTSITI5E/awIJhSz9c2tNdEWfbYdp/rpj4SAKQ/5ZvZu19NlsxmHrg8T5SWLv7e5uoJD4vdl1RxHyAypyy42wZO8
t5oWvJWc5w7PQniTM/kcbHFPKMTlh52FWUkkd+jcsP5NaxPGodLSvhXRgXAU35L/fntKMU/4ioeXH/NKF/9Pi/dE2wDTrcymrulY
E9GEOEDc5LS27c21rWvmXa0vGYL1qPnDkfCnW9Hqol1yZ20mEHICdnkIsnPJFbHuD9wLBtCkHVAkoOFRClZLdPWWJ+W8wEDoLdyi
lGr8qGherrro59bcRXbTaRtjprvkkc5ZBpTvlUatmh9S2GS1kHZrxcj8LHbmsAHEdtASUHTNSh5WkbVtQFuXe0UAgnUrrldrnUns
ZpwyrfQZR80N4V0FxRq6P6z/EmjydjjMyniYlnsDdtZlQV5d+Lxzy1lT70q8T0jWuQouM/V2UaNtiD6sh4+3CU/qO5cfZIWABOo4
1yszjRNrboyyKuDkUNho2lhIiemTTXlybOLyg146DPeLzXsdAuM8z737cmWBJm6HcfZdwJFTHqEQw8YE3YHAw50eQc+QrXpz14aM
5VkOo2k+ZTviI26mWTppHeEle5m2TusIanXZhxXAMZs01HvFlra2SJzZL2u+DLCi5BtKLVtvlB+sTAujH2IS2dpcjlRdEB/vndew
6weWFweFcrrEO9xgM93BuNPMooPomkeBn1pel9Mqr3Bvize3OQxvAYo5VcsoImUMtAj0xk5cqe1wM8Xug/Rjs9w7arSjFeKzHdsT
QQ5BTvNE2QKLSCPk0XRhp3beecLsP+pZoqwuP6t6X34Yr7iaHdAo1t+vo7aZXVWAeUEdCfeqXpvd0vvV7jPCdvVhr+TYf7L7zKMu
p09QxbY4a4UStlHsMOgaBgaxj56XPdwFmEAqnzMtdtOT8V5+2Es7kJ/t5nttRUQt6s6z/oKkjBUgIMtIRNJvjYTX7lWWTDvpWfWR
oIrVa36iPS4DD/4X2hD8jAybZEXtnr9A13tEySR9EEqtpvoZo6qVuL6avSyfoHi+wfW9Vmed43YRI7pV7C72GHwpread4+ruk9Iy
s8/Oklwdwku9lZ9G+2xlLw/hlb7cz6PNQe+1ytdcdsw9WL9CXJCAOeruvSXeePS4UB4t9KPi/jDGj7gN0d1sViDVqE8jmiAnwyZR
u0WbuA4CF1D7XYcjHgIymUWM670DbckunOqsfOt1ECIDDa1hjxr8nmK3++5cez7KeVnmw+oGqd6vjQRSiUOzD2h2RhRpW9Hj9dlu
5fTb7gCFkhc7c5rDI+lpFTe5x2ZaOI/wLmR52NoTRCnAi+2mb2uoqCGNtX08RNWE7ElmeKV6L+tJdiiwpCIuwe6d9Y84Oz5jtxLw
J5GMT+Stw07a+EdI37gxQIN80HsbVpOB1jBjaB75o0SWh6Z7O6Egs3rCe4WTv6WciR2zmZ8cGLh82/uLF5lnR6w+OwF29bbi/w97
JT/j4RcnileDIFlHThFE/2p1WEUPzDR8Zh2lQZbGbP2XObUa75PdsasPegUGf7EZ73V2Xv/iC5iqR1A9oj+YNCY73evYsg141hG2
y0/6Prdaa7A7LLRsu78rRbF+PGuUNotWfbtlUbCHOBXn8e5Vpl000CETIFBew27ni3Vv6AMa3T2MD2M2VeLhFghqQOxFu3PNPE43
i7ZKtD7l5nZdslvQ1vZh2YyBhuirxnvtEDGiKL1AMJA1O9sdPfy2W0fRXD02gT3PeSii3Jq1bKDW9/LkapzLmeOVPY2fSSbHs+H9
a03talX9tY3PHwV4s6vx3slJxPIUYj5qq87ZzUz+PBbfnJIFRkdK7hD2kc+Gfv42PDmwfXmt3D+ICrtIuls/oIwC9SpErYNIdTFO
FcRt5MX6tKzWvuDuHX1JAVxD+VS/ul3EuW1LuoxFBOPSYWRwu/tgvURZHxV0IAWGJ9s4l08J2sV6CU7fyADWIAY/FiyJXcTOS+qe
/MgqN6igB2NnJmqxO4GiJKyXXJ7sfv4pZdqf1eGsPsoTBvGn1Go/m312pfdLpUQS38eF5RsQGAr4ZhfgvNZUhDsOH9Cq/VuJ2C7v
Cv73WCzpbDw5TT6pDbzgUH3v7VFAvniIg8Opmgd2x4TqDDt/rPh0jdO6AMVup0QhOduGSYwYUlqeXFJ0+Wgfseu0jLSbffWE3ZpR
l7WpzWHtVBvC2ralgqaYh8ZkTcmLtw5a+CGvxSC/8oqrN8tayTPARmFIfll/ZYKDtDigXtKhD/1sEdxiZ66Z7GJ3XQb3pG568SFw
NXfqXIDDbklHNqUdG+hlZ4J2jXb7yiInph/2nnXPvnADwmV7H1fNnBbrrVseml0nGiT6Zc5TYHKq4hYatwY3kPEwvYXqOOp5O0Cx
K8uesJfLj3nh5X5aDE8q7q9cELKyt+I4gm1ao3CZqIER8sdlAd0uHjvPthClgpggsebTbHxSgrj8rJfuh7ej/RVZ/g6zL1z4H5J/
vMXmCxd9x1TcI5V3WL18R7xUPePztsXrV7YnCNVbTF6/mF7ICeGR3mH0hW8ZIMD1LQN94WsjIlnskf07rL7wtRHnXW1vAYbLt8Uj
L1J4h8nrV/+X4t8zyhe+N4L5L3bH8BusvnD7/9n98b8g0QsZJpcQoh2w0QD7QvZuX31lzst5pM88B/Rf+QAHP2Y35XrvOv6wp0e3
IXeHeoRkn9jJsZfVeHQaLZ/3zh7w1QLB/sD5VOKtLzS9vKb67ctFsQn9TLfuoQIHpA5rI7LjER5k0F3W8oQGWWeYnGrS12EdPN8i
tNRa743T43BgT5bzJDSs3Eh+8GguT0phSGshBc47BuG5uC96HstJ3BO9+MJl+XUjCH3kNc6zcoi5bC2XVovFgYNs+0es+Ve4t47V
9Fturrdbq3/EajY5/VuuWf0ns97d65m6ioUB5E/lcd5HzAC83rt9p/loLotgmjHlYbcrQo2tLbbbl4dNEXjFXO6o1QTDUeJ5Xdm9
DZq5iZ9c10YerqqyZ674sV0J4QYiyM67kElzTrzbw25YLBLLs5tsLz8LEK4DvSIp2jagapAw1YFKAfe3Ly0jt0HarOx7mj1N1me3
BF99zitXdv8wWW/eplQjP5KKbPadX80ssdthXLuJgczjgl3we+7i8qxs329DsMIWn34xytVn4b5o+dL4AFQF2Fh2j401s2tpXled
M+8+nWnSz2aDf9Lxd/VZr90X/qvZZydQ/pRJzmDLN7OaENty807SOU2i55bsCqvu4byrFyXJ241Cdiq8dyR4CNUOKwgC2cWU7lXY
r8v/ZBMAUpx88db2hdj3RcTQXPbbzgVBr5I1H1phySrWALkkL+f3/BXkYslYrRLvHcuAlIGtM1n3+A69OKRiXhl1areTbf4JR7Ms
cd58Xs4Se9VsLT3/8n8BsjLAbjx+AAA=
"""
LABELS_B64 = """
H4sIAAAAAAAC/22b2ZYUR5au7/speIBols2D7hhVSAJ0AFWr685GiKUkg06SFpyn7297plsEKmlVrVJB7HDzPfyDmcXtt0/jUHq/
GZ8/H67Lx/EffXw9qK9a/jHe2ZBjNtW04rtKrlQ/RrE5tpJmcake9PG6fXg4vrYP5fr9Hj3SnFP7yqe1U9nX2WdJtSbfk20tK9WM
bT4cHs2b09Px9T7M+Wim8d45o2sPro6pmzEtlaSTUyN1l0Md5fDoePP2z/LpPqw3nZKZ1qvqTGm5GP415RxYf9VZ+xZTNcodHl1d
1ePtA30fN2eouvXZTbfDxhxjj9mmUlSooafcQ4h+mLjHmfs4FlKLUroko6dtQ0XrYrBdqdRyjy37OH3y5vDoY+njy+c346p8u49t
KadZx9BkwTU1p4pxlsIrZxZavApjTpvc4XH5WE+nB5extimjhiKjIfcZs+puuhEsL5BjLY4vTcPreXh8vG035fbY9sA8a9fEta5n
M6yAAthRyLD1WffBUkIcUQKffRsrQdWXXo2106SWQhszZOu8rnxWT9Od7iXHbrTEnb4+PJ728puQR8q6mNESmSg2JqnQLN6yDG2U
H9mMfHj69Nnve+2zy7G4nrUrgzVZ10ZSyqo5glEhxTotK+lbzAP9UN3HlZnKCNnTnjHGUGrIavjgawmztphr6sPSvIen43/Hzefj
8+N9YHS+l6KdvFM1cdipaqi0bQnDhpprVrNqqw887+GjH3948OvN6eu3B09O17c3pd3uvXDxj3d6GK8t3ZdVab66qFnRdDOOAx9/
WK++7BMSg1I03KiZepdgYg+1q07GfB2WsdE+BOvk6V8f3p72VjchVd4uxO74YFcstAeVugpKMyzO6aCYHjL74tWrH1/fL/o+2Kfi
s88lNE9BffdD2RlZgddKRmYYx0tMfXj25smDfq5MGSW26X2eIbbRWyHVQxvSXptSdPBIbQZ/eHb7Ydw8HVe3ZXVQ6nxOAxdV0a3R
xaInT7WZFPk2TPGthaTzZew+ZtvfxVwsq80+0A3WdJKpbPShUe2mB1Vzd7Evj18+7kkqqqleR2txuB58mi4xNnyW7yLAKQ/+qHR4
sbXS/rhYIqDRs3O0nE/NVVU0D62lmumyqp25zOMual+kac2ZDooEX6aJTHXOgz6Y2XZdBBWaoIQ8692DZ9+jpOJdgkkqV+McADTi
7M1oVqAybdlyysCkM4efTt8+n/6j3QX56XUplZFopdDvcQwG0syUgEFFiaaOowFaj588uY8xvXejjI6uRBrUje6NGRHEM9UFXZXV
3nvjAbr25eP4el7oXfxUjW+uoIgxYHJivFsNk+YtXmurTEolRIDn0c3t8fODd19ualnfQYLvviU7n2Oi8DoUl6ut2hsKGN1gHUHV
nI3L09e//xZz/y2uTz9t0NKtUdNFhQX02Jni7gE/27vUuh8evXv04vf7IGAsAnthlK6ZmEqVqL9K/LuzupFPm0D+cXg8CuNyF0Sv
KhLZQyoKpusFWChRjUwTwRjaGaujmuXw+P3p/Wm9Zhu6zg2HU7UyWy1AXkPJywc6RY3C+tW8D9vfa0bbbAO3+tCqThOaGaCH1QqI
V7k4nmXGZInHevq6NwMM20lE6pXuZBxCc8rwPRRamUxlXGkzwibHPj7/sR5GA7ReVacnPMQYRpg0WumeJWqbrK6ZNfcVZ/c8kuqk
fcyaDofBHXM4aoR+gbERIJTEQqcQyfvXr57dR22UZmlRGWX4JFcafs6soWugOfLoCHz2w+MXP7942E4fVy4nPWq7oRI1BphaOwrZ
kh3JtBgNr9ayyVrxvCsKd64B3NFBujk7jQGg2jSbEK0qtEEzdC/g7/Met+fFTkDJzzqHHUSjc3RHsbCEUEuuvmUboAkpwnW5bufe
ltaTDA6Uy7C18elcgPPQI7WObhqjMwS4Avcn+uBMChaKy6anEbcHeIeEQF6APUbZUgDdFbiXgvnXnbozAbDdHKlYF3hBE2HKCaVT
Gbri/ES353TAzVHlbik+7aWi7xC6HfwBY6Bam3B8SyvQ75PHdzICA34Z3o/WszIeCAiDpkgObQeYgQMrMNwH1kGTCPBSQlQLIsQm
r6GfOR0oCeu4bqnpCoz7UmHIBFyhdIpzlbezNRhGXgUwJOUI35GctgLTnhylUwgFirSdmUBfAVSI2dnBsOCmp/+8i+cn5r3Ba2JF
IiZ5nzZ1D6110SIQIJgOpTfYtrgV+NOzHWOcQAy5Y/5mowLKR9SAA9SqNYaFwpw6Zi9i6T//tWNMg1EEE5A2dszqo8jeiIaZxSRa
vaOiERMSZHY49x0GNMxhMkwiNYNRHXLLzwIuAzfADsyxBT2+Ol7vaGHSqCPy/WnmkrzIFFR1cN7o0C1ToWaP0VWJG4/f7Y9DiiCD
UIjUbwBkg08BZOgaRAwiIJTMvKhNds7jNfSxT0WEGq2wOEqmKVAzG7RgTdSyOe/oJOYS6jtH7mORIlhd+Ttd4QhffVXYgtm9MoXX
FfEEh89yjlwQBUDAcyiTBoCRP4C7+96qCbT7bClCZSDIivzhwcsvV7fHt8f3Z/igUUdspBHmR+DwsOmAPYXYBOjIlB6I//R337G/
ATQFuhbXkN16ipdq9BFzltG4CFWrgPcmUPni3fM3r1+929888TE0iAgumTgZx2KQdMBD1hOg4A+z27L94cvHuhZNx9NtBQzy/HUD
hRWeQDqxtKhdRfUAhDOfA9W+VBNt7d7hhsR2sWyeOHEyiHy0FOReKLCu58j9mVZ5g/SpTAbaMxf6aQ6HbbAJYVGYGOd9S2NF7tnh
EVsf1YIhQE2IocGOtU6vNO/IutGMoV+Bq7xYJj3UtArmQy73BtkjQE0yKLSGoIl6MEUrcIe9hllJGJjhUoK8hhWpjPxSSGq0w5j4
OYuZWoE77FXnECcMnMeRKB9srSGNJniukqVQkV6wqa3AHfaYz5iAWgSun8DRwAMDrbjawKrBaq8j/uuc1h32rEbYI6s1thXEdDMr
2AS51Y1FkzJrenPiKzAtWqfhm2jKKNIR0Vw1YhR1hUFgcMJg7Oo8905evFdxdHAkOcULdTgJbYmYDDS6iE3kI5Zx1fGHZYNW+0Fd
g3pXZBQaYNJ9o6NDfQVAgTDBnoqI2rziy3JzuyM1nUdVMFrGwB9oDwwTLxpF+QOMBdCHnbau/VjO0IKxh2JdiMCkA2EwfdbYCPoh
B3A5sGOYMZg9zqwtBvC40KhAoK25s8IIPwCLmBLIbWY0YcQb/DL6e3Ek+94EX2ych6gZa3SaWKfRFKiQBj1cqkOHBQ9S/3Jq5erJ
6Xj94OkGhfcWHKSvrTCRtU1LbjGLqJ9ATrRKk14c1fkS/hJuzj6qAUDa8sIaNwMahIIQAikqMzttxT1jyy7DcSn2Phw8CeAsr50H
OA43iO9kBLA0Fl/vU0CHRQk/fbo5Xr8n+IcHj8eFk8N0mLqRDY+tXhAYBqeswVA5dGiQLR1jDtfyYCr8x9g9csxIWDw9HgPALxN9
CrU2rzDHdDeSCYUFrR5el8/Hz+ctAb6N1qGNtcj3KphYEe3IxwSWAvIoZjCyHF5/GteXGyV0q/PQLcibM/8dqC/QrGE9rEUFIEKw
zN0ffi03pa8Szwywlx49LngER3HcGEh3nBWqdfSEeIQy0xZ2sf9UjBepnwUgEJCVwsJUVEoET690R4gKmXN4U3q5+W5LR0/VNYrf
+wSblx6mtrQi1Bq1ZgCAT6G8eXhbbr/cXD94NW7/PN38cTatxaP88UAxFp8SFo1keuVbIT+KlCGYO9X9a/zeVzwX2ASMVOIblKaO
9JRoZoAxBzBhoL5zPbz9QEW/W3oGfHWwdL4S78lgTBy+x6bHWikMqjSEmdThLf305arc7GUV5UpacYO2KK/poST7RBm0gIRo0zl0
ntMc3t6SLeYXE7h30oCWEZxIVsEbjBXFof0TVqbSJcwHFrXzzD+Pt+3DOJ394z/3V44gd9D42KgsMgZVmRGfGpVIiWjrNAFzBPXh
3Yfx4HUb5XqvVRsRckXp29wQr2hK/H7CEDmglaUF3Kzt6fDu9Me4fvj29nSzO31PhVVB4/GwWFI3ZvqMIeHrXIpViaoIsiG2hf5U
btbOZmjYJL9tBsw8gTTUgwJm0c45Q+1Mj+zCbnFXp32peCOFFAezgUNkloLl6uBloZSBDg9a5scE4m6vhuwSHT+Wm72ywG9xNuYs
Yjh3GBciQd5vWqbiAWhZXG07/HZ9/MwYrIHtgkEZMKKBNM1raekGjWjAQZtEwgINhpD4r3EGmF0SYM9RHa3k0YtC4uHCYsIrMu+z
8SJjmGJw9zuiLyOkMPRkCDfgp68J/SLblTPbjILEyZZtj2GL+3xbPn46KyZvGGhVA2ksDbbuDp5N0GcAKxKsREndncy7i1xrZSTw
S6MqOyyuXhodcKBcscO7tVbdGph1jlyr9ZnEoqu2ElRUSyMxUOsIEZWbIxBpTNTnyF3AIDRC6AOACFO24oD+qfJGnUxxkv2Mxriq
c+SuYCDhoLXoUSxpQ2VZDDvDmhH/2BLWa6DEO0l5F7kkTE2C+LhKLGiBRSiMAiNHLUYK0mFPBY2JiJV+L+ctI1JQo22sEQHupiSG
GBwEHAs+eqP4SN5ye3tz4RkGjKYcMo921fiwEPAJ2z50FYsTMi3g57QrcK+J1MCoAq2GhDhEORU/hpxbDPomiftzBtpdgUtSFiDQ
iQYAv2XvTqnNn8hjnQFGA06ilU38/Pbq4fG81TNhEdlSQwULADXRn5BkNxowyXwlZZZinyPXbg9IHfEUVo4nIs2H8mpNpB1ja5k+
G8W1bLLnm2yJrGZHIuNjSkkJI6F4Y5gC/5WNRkBnvK1JiGqJ+/9X5XbtGoqadNGaiILl9SrDBf4la6WGQlfFi8I7PL46tT/eCSt+
/usuIDOB0Xeqyb6jGFenGXYXgETMccWf+EBh+Y53T/7z0dWnD+Wv3xBtRmVI0/oRMeM60rkGvdaci842vFYN09CIf168c8cG0/nw
taX6EbOhYWNlZsRTGpQN3a9Tnocn5gmg8o/T7YP/KldX43btGAKeDTnpYTTS2wzjlgt0l1AwoqyQGQiywxM5cnmyLfhmuTH4vtOy
2ogqdNgG4L4KRiFMGQTq7qnCd7GryogWPCqLNnOU1rNGRwzZyVDRatETQ1HDbon+/OE4bsbn27X36AMjLr4VusEOmNCYUSXST4ON
M4DGzAGRt3+zX9uDS41mBP7cQPhUrBLvWzCQvIFs9ODzfDiw4OP1voGRw6Tr6S0ZcbwpnhuQw2d3hLkXAYpvoesOd695dbVvlKaM
wtBbL8FiBeTF67cKSVoUYnDBtOxzVYcnp3q8/nA69TVBVbwJ8FB7s0O0CvxY0R6ZiuG1CRrRxHEZujDY489hfbfZbItjR87mLjlD
wgh4w7Bte+rxuo7rPTnWBgRsZsRQd7XlCo6XJMaHhYD+U7bEnbTDfZyYnav+fU/VUJlwikh63WQOFfQRQNWACFEjWTSRFoUpX9Ll
WGMPxCVhUD1wjBRGlY80UwN0EKo4aKjAI+XvVz2+LrejcHGIRTkokMr4GRDbKjshDwg+ihoZY4vam0HO4+4nyNEA00PcylnfYwli
8RVJIEUwMpSnJw5lC/9Q/tjj5IwP0kESI4ht8SQLvak2ZRyBKmytkf+suB/I0s3H71NVwBwcj5yGFIRewC35yDyBuiiDYUBq6j/7
9iUvx+3N6d+ABymNRrMVn2LQlRPT3B3is2D8FWJRA0gsdPuG0/V5V5Zvn212WsxKf04NGaCeFJ6FBtG8CgOKfl2Ba/NGDgeQIZ4h
wmdO3rVbC8LT0y2hkpthWHzeAj9vWnMH2h4qpcCzI5Wm7d4MZBDGLyafEwsds08w9PDk9du3l1zS4TaFi0iuRjAGtCiJD1KvwjCh
anQSCO8rcO0g421Q8HUGH+HMjmJMMsXAOw4i0delRQ8lPHnz7HezqwlfC5CkQBQNMPho7baFa+TwO9WIbpQD4Uxqbr59uj19Opa1
UsMaPRK+DS9MlXEwCEQUMAAO1ok6zqiCfBG6zh20KPiKf45iiVoIvDFr89igiS3F9hcrG4JPj++Pzy8GAJJg1ii2ESyxFRoQYToC
k9QQQG4O6dFwePpybciKxTRQeEqFV6PLa6b3M1OeGgxNXRzyb8bDs2M/7S2fegwamQJz+4SEdU12onqKoiW9tK8BZLIcdo6bWj7v
PYqRVnKm6vBTtg7txCnSOFMOOU21/PFIo4bDMiSP9iYFrU2IZADsCbBoYtQrAEhmyxTVS3rdFvnxol1IACLUzMA6GWMHwaMYCrTe
IbtQChBUrbmLWphJnhjFrHuBTrDyDgVK8SZcqjHJSfOSIMTheSkPFxkxKeRN+ymnhYjb4BE9DJIcqzv6p1UkYuzz8FzGYZ0WYN2H
RhLbGapQn+z++5wtyhUxJvtmGYY6PD9+Hf351amcHzfwdbJRzcjKFQKRFwYG6eJc6DIjx/twwvOrb/W4NpKoE30pXYTAQc3PDazk
MMQnwQo4z0bTDs9PV+9PN9/+CjJGNucoMy8kzRSHQSwhXqcrUsHAK5PVbg7P3/3+11hEQFQxB3rZIF94bu4dliZfNCCdhr7MaIDD
j7D15dDL8VZDN8G3JYPtEZjxcsulQgOWBOH94JG5As3SYsAEjhJBI2ichdubiGq56xJHFYxCW7UVaFffoEaS7K1iskRFyf4rgjdG
0MbjXSYmb+q8B/7NTl9G48n1F5oU+oa/LP4LVWcKbwwOEW4pO1/x+Pe1wywetIxUEYpVg4e84oB2mDHbopNzk5C8Pfw4Ph6vj+tR
gCwLxYyTRjmxZIb5NOstTqGrmgre0F9pj1teAHXVYLyKBUFIzzk8SAqqbeRZtTAB/qXsccudAU/eF7nYxKuhyMPIPUI6uBDch+oD
NYanuI/72+RA1bIhhPpvfJNcOqisxsgWMHjE/FiTM8n5EZl8oXKz3MDRDpHtYHpNKNY1VblUlJxOWFEsmwFV/3G8JXI90IuuVYlv
hdUii2xgDf5stiCtjMV1Yp7THrenSHY7ZKMZHK2Bj1g0Bilrsonb2tCkMJg63B5nl00n+RiJlmCZSX1waXhXJxdRjNpOyAwGfB7+
cTpdvB4+atpGWFZexyz3qphnkTCYs4kkaKii2SJht3d3tO7x2FsE6bahoIBPO2TLZWiatml6p+OadWYw97j1etU3ub6FzBgJAQGd
yTY8CZFzKOxXwF4ZPO8/vqAsL9wnRKiRgGgSdH6nksAwz0izdGQt45PlLG8P249QZnF1O15BDzkAFfJtzmCKEFq8uhuwnAGU9ri1
jc0nvRx9GJAiOJQs/1fJXqosG1WswEhbV9w6qDaoCZNZG3KjAsFygcjHXkF3clzlNNFBYXvcXj0R6nL+JlstAeUkm+sJ36utnYKa
NJBJKO49bpcLedM7MdQ4CywPJCKRZAOAVpBo0BElkNY6960HDHhQocjhB87OUalgZL+8+lFpMdWLMKtfZdg3HuIQkSdyCIukg+CZ
EptbBGJ1hoKanFq6PW4/OjFGTulJGuhvcbQZI27Jajc0ODIYQQZUz7DH7ScneDFMAamOuGS8JivGJeApjCqpFMZEDlW03eP2gxMa
P03PFyaArzYQcKTCIFB4hqCmSGtT3Hoft3hYgSsdIYEDLUPu/sj1QRwPlWc2cPYQKjXew9Q648uashUqj8HC5tDPxSQgQ7h4ExjB
qrYed8ZRx7AgkXAQXneNQdnOAnBrskmBLmUcUDB73NqmoBUnErQWXL+brYujRMOxXjlsZPxRqNiIPW5vMy2oiyiJ8KjucmegYrCx
Sl7oIlZTZbezrfdbqtQpxOyMHc07dAOaWDjp6QEbChMiQbAxYexxfp0iOXLnqnPdJCynBWt1bTZO+gC7zx8ouYG2x4W1GxdjibBP
k2O9hghFBss2LfNf8Bt3gKbW++1txjdiMhsV1CC9Jx2zsnaMKl5YZ4rYO+6y7XF7m8UsCyyNAWiyWW0y3QYX+VBDwqdYufmFAtvj
8jL0TEuG7K2Onh620lVJI4UQgRAof4GfK/v72RUmi0q19I5ksj7KUWCzVg6yqzbRer1x9h6m1pFn60r2NU2viFIrW35aN1NxqhMy
8nKY2kfe4xaaddKpEc29yZUXVpUqjiBLe1k5ilR0OUZ3j1unwQmGtl7885SbEwFSA1yGFVkiJ54OqLLn19vfr+Fs8I9DJAeqJZrU
Kn4PvGV6WC/+Gs6ZexmsW6g0DCoruSD/I/d4O/YCHwWVmYEIC3I/rO8ov7pTnNVI1BYrInexchyYQ01vYm8drBstoLxP396cJAPp
C2AzDKow7zpqL/tgFKDI4Yx4BtkFugsL5ztroh8Rzz0HJBdJMbB1gXiDzh1ViU+Za/b21iwpBNVIZA9ynUtcIbjW1EbUqJKBpqXw
+wjtnanlgFh2CCHanCZIEXF4AdmVoCYUHl7bo6Dvw9bBMcq1WUClDZsi3qmZ0eWeVkYmoaRSowrisV708vXh+6tTLVd7NrWc9dLO
lBnywxUGzeyMDKLOVqafFXQ0+vDiup/6xdFvbltDtyF6KZIhzKZ2tFirDF2soC69jYTdA/c2k+NLZ1EcM07K5ChcgT5TRiSizvg2
atR72wKffX3wy7unCwfBOijLR+RqSkllvCP+Fasy5Go1Gj/CMPbw0693J58PW1tuWSvsjlwmKwKjDECv1chNFoXflzvOBfA3h59v
yh/jer0lvBPQnhkf702L4K6kE2FXKyzI0+XkcMBI93HLYqOYmthxh3VCv0HaJakIuMzeNpAoZJns3Mfts4QFxbYy2DiKGRkkB81r
CgrJ6Bod00tPNbXHretkyE8UTu8IRgiZN8GEAFbQF0lmLGtDgda1zqUMEgbE3G0IkQRjHc3WGoJiIpqZMd+hfFWJk12E67NpAjVL
BGSMxd5DhGFgz4BNPDTFqXAoLRgp44pcqonWR/JCI4FnTywlqgWga8xlGcgmJxd7nDtHruxUL3eY5ADBJwSNz7IPGwf+tzuRemHI
3lQ6/PxlO3xfa02wnFyEoVH7zOglJ3cZVUIzJJUciIcmym2PW1UsSbfhpV6zaCXX+phkL7clvBjMZEE5X+V51+W80aeABmbHA2xG
rjHSqSELjjOlycpc0w8otsMvj7YTw7MRRVhsRImgVJhoGF6Bgs61hLCocjxW5FbgClxGFFuvpZJyNwrHYgyd412zw1P25DI2i9HQ
K3DPaGJEK+6x1SE6FqNFhuU8DrZJqPWMrGAN9fDL8X++HC9Nsx8eXc8HZp5yvVXJDWDcAD3PBDsjIOawhufI87VXsXUjVNmpAnuy
qXPKzy+aR28F9CVmG3tyF9n32jtbaQ4j2kwuWWZcgdG0DeoryCV/L8ekk1c8/u9oVPHh9dr8HPgA+hRY07CKeAoRAArrrUTf48x8
k72pw8tx08rtutgbN56Rs0LGDnEBzeTYNBLdTCFvOzVwh95++fvZCspOlfwepcOIVss5ou1WEjQE41yUc+5E5ragZZRQhUqOULo0
l/YitZycQ8GhI8gpfWCCvT68evb74xfv5AcFe2+3xiClJBu6E8ePsJHrmoiMwoBm+bHB7CXWw+ufn53R21hIi5cZsgsrm45bIQcS
08ktYnyk8GKzd1FrlRgCVgpdG7mlE9N2UsAzxdZF4A68yJieu6hl5+EWOSeHYyFm/KoTwGHuoqqQbkOHdRCjHV6//PH5i1d/3dYp
4l4mqFSimXIs4gswLNhbUEKpkbhc4ePD63dPztcla08DTaAbDh7FJX7AQdiFgrqhkJsIddXsdvnkWL+cE1NrjkwT5Z5DxFpQ2GVN
PjXvVsUVJrpm7HEXR4nKg5RIelsD3BZCS71Y3++vwsvtrRLSHne+KIlIkwvsOIkEs3tyw7rIihgEBb/KDxmQJfdxbh0mdNQMfcEs
JTlLwIJQHlRv8nJzDIOgsOvz8OsotwuZrMEdOzl3mbKXbfgSJ5ubKBThbTBYK9n7PPx6ujpdHy8OhNHgGAm58EYu0Qxt+tSK+J8o
dwir7LFqUn6OXHyP3Ia1i6rYRpflB0UAjaYlGCiEQ8weCAgXz1y5AdxB69LkggUYXcKY2+8n0CRBrlYgdI2p5RzpzofJE/HZp2x1
0wIGMMPGIrqqiVGeztA0W1fkDw8eP3p33oM0DFuDEx2Px+rm7QppQcjK5iUsDn1gKy6C18VRiy2Un2RUCl8FwuHuuB0HGGxaGLhp
PEfTF8FP/rk3bZBL/FZ+LNMAJDoYJEB94agAgSH3+RxsZONF8PPXj17uuh8VHfgKDBR1DtLzydAkiHHoZEY0eA6hz4voH1+9Xh7M
BmiqT69NlwtTMCtFzqK2ZUfcV/kxWDDfBa/bsh1WKsMPfCWaumS5o4NsqrIEOdpRdJVvl+n++dV5Us1osQXqqYKoeZiuy7DnkXDg
nVEf6NfLjP3y+vX+0nJpWyUD0Q2xETPJWSKmx8oRBvPA64C83z365aNX+65/8OJMB44J5SFXqCaUL2iTYAhPf8nWkxx8nqNf/b4v
3AaRRmHbjZGrEjgFpk+uacBRaTsjxk1FdxEMwq25xwhAJHIk3eS3ZzBUNxaNYnAzIsFwXNvluBX85tmva7+DukxkqJEba6N0gAx5
Y+WeT3Ba6YDqV/OyWG9XsZhVZgi2cHIzU24zwv0GqFEOjz3qHMzYKOMy+N3rNz+tTYyEz+etW9dxOocwgtZ8pb+8AHFwYIHcGT6H
//b26ZOlqHoQCMKoaJ+dMuCWztN4+UULTWqhf9ljvoj+15u1f41FkeNmuQZNoQM9CtyqVlDJVQ5mEU7Nu3j4f1/K9W25oDu5iQd+
mUlDyREq0m2Ohu9zWX58A2BCRqrOc+QyxXLav50Czk59gJBgq8KByi4anstBKXjCdo7c0QtywqUjxjyoMbNinFC9EFXL5EnJaa/c
G2mHN+Pj8bZcSGuTdG6iUchQoCZNpHwl8cLpATVm5Y6UyefIpQMhRQOtyi8Cs9w1lTO9Ch9V8DajVHUmDVj4tx/Kp/H2w3Fe7JmP
YHWgohXGHcOLJUdcy7NFYhQXa5mYjMvYdcVCId/kZyeYxYD5Lg6aCC7jd4b8EpjsoyywSRexazM0yEEibyZTo2TvIsqPJwrmjm7I
NIwm71pfxi4WtKgU3JiKxuFeqCCCJ8m92CEXdUMhFkabl7G79aGMtQFbcv1YDzmjCjQm/0KKUSlJ7sMwJ+EyNqyti64h2iA/OPJI
UW1pTFSBlfPaFjNiZ4gBu4xd98qNA85aRj6iK+VCSN/uuRn4yMGpoJj8RNAd3p6+3H74/TstJD896BnI2n41JL+fAqTsdv0FEzqm
xzUA3v7w9lO5uTn9+e8//0N6IjGAzCC/f3bJ47gmXOp4A/gW3Y5SRnv++zcsEYgUhSyw3GIrjByLYjK73L4NYlmZ4Wjl1253tz/L
OovEMAuDdg2+y5X/Lj8sjjPKpS3QAx2JtZb7D+8e/fiv84I7U7KtTsnxFJ7UdMgNnRqjNowxSCgHmndRy2MGSYf83oPeSHKfXA5J
cmpwCQlG2jIMzZvDu2O/UDkK0eDlBF0cEA8iURAf9IHRjHLhX+5NGLDxLsysU88gm75yGu9ZHRUCqHTsES1Gg2o5eMTyebkgemqb
q91BwiB/YVv5TUwucGiRa+kQv9zto/nQ5p28dCI/Xf4WCCULVcjNialNTdt+pkcwkBZwDYTh5XFt2xWyszE1fBj3AfC5bCEsJDkC
LMvFboS8/MBryk9t7OG3deQvFwuGCaK04GmD+m6BCqKAWDCPl2udTHs9/Pbp8sAIgDWQOR9pctUiRDLkk/yojzg0mNzhnnJ6fBe2
blHg1AxOT84H0MWMKDJGNN8QjTAbz7ao/nwftqAWpGg58LdDLlgWucffrVzvkiMZMaOYMQcf/vPiMsP0+ElHgSO9oSuWBMTt1Rfd
0M3Yoy5nfpiZ/z7xrAs3mShRlWkNXsN0VdxECJ5+qYj16mpCdwSrDv/9+vXlGWFH+2YehU1qsDVYhc9oSW6TMkMIaBgfUcoDv1xf
HIY500uSSzA1DojI0aCmwmAsecr5MarYyA+c78POkKxmyNqITbdIEaYP7yb3y41c2qdNkf++x/uwPZXQi5LdQpRTkvutiKnoeumY
kyHbHgIUMF89/Ovxdq74f1DWYp+6QgAA
"""

raw_text = gzip.decompress(base64.b64decode(RAW_B64)).decode("utf-8")
labels_text = gzip.decompress(base64.b64decode(LABELS_B64)).decode("utf-8")

# Etiquetas reales de exchanges
exchanges = {}
for fila in csv.DictReader(io.StringIO(labels_text.strip())):
    exchanges[fila["address"].strip().lower()] = {
        "nombre": fila["name"].strip(),
        "tipo": fila["type"].strip(),
    }

# Transferencias crudas de SoftChainCoin
filas = []
for fila in csv.DictReader(io.StringIO(raw_text.strip())):
    filas.append((
        fila["from_address"].strip().lower(),
        fila["to_address"].strip().lower(),
        float(fila["value"]),
        int(fila["block_number"]),
    ))

# Reglas de limpieza del paper AlphaCore
CERO = "0x0000000000000000000000000000000000000000"
vmax = max(v for _, _, v, _ in filas)
umbral = vmax / 1e10
aristas = {}
reporte = {
    "leidas": len(filas),
    "valor_cero": 0,
    "polvo": 0,
    "autolazos": 0,
    "direccion_cero": 0,
    "conservadas": 0,
}
for origen, destino, valor, bloque in filas:
    if valor <= 0:
        reporte["valor_cero"] += 1
        continue
    if valor < umbral:
        reporte["polvo"] += 1
        continue
    if origen == destino:
        reporte["autolazos"] += 1
        continue
    if origen == CERO or destino == CERO:
        reporte["direccion_cero"] += 1
    clave = (origen, destino)
    if clave not in aristas:
        aristas[clave] = [0.0, 0]
    aristas[clave][0] += valor
    aristas[clave][1] += 1
reporte["conservadas"] = len(aristas)

# Construir el grafo limpio de la sección 1
grafo_s1 = Grafo()
for (origen, destino), (valor, n) in aristas.items():
    grafo_s1._agregar_arista(origen, destino, valor, limpiar=False)
grafo_s1.etiquetas = exchanges

print("SoftChainCoin — transferencias crudas:", reporte["leidas"])
for clave in ["valor_cero", "polvo", "autolazos", "direccion_cero"]:
    print(f"  {clave:<15}: {reporte[clave]:,}")
print(f"  aristas únicas : {reporte['conservadas']:,}")
print(f"  grafo          : {len(grafo_s1.nodos):,} nodos, {len(grafo_s1.aristas):,} aristas")


def perfil(g):
    """Resumen estructural de un grafo dirigido."""
    n = len(g.nodos)
    m = len(g.aristas)
    densidad = m / (n * (n - 1)) if n > 1 else 0.0
    pares = {(a, b) for a, b, _ in g.aristas}
    reciprocidad = (
        sum(1 for a, b in pares if (b, a) in pares) / len(pares) if pares else 0.0
    )
    ady = defaultdict(set)
    for a, b, _ in g.aristas:
        ady[a].add(b)
        ady[b].add(a)
    vistos = set()
    componentes = 0
    mayor = 0
    for inicio in ady:
        if inicio in vistos:
            continue
        componentes += 1
        cola = deque([inicio])
        vistos.add(inicio)
        tam = 0
        while cola:
            u = cola.popleft()
            tam += 1
            for v in ady[u]:
                if v not in vistos:
                    vistos.add(v)
                    cola.append(v)
        mayor = max(mayor, tam)
    origenes = {a for a, _, _ in g.aristas}
    destinos = {b for _, b, _ in g.aristas}
    return {
        "nodos": n,
        "aristas": m,
        "densidad": densidad,
        "reciprocidad": reciprocidad,
        "componentes": componentes,
        "componente_mayor": mayor,
        "fuentes": sum(1 for v in g.nodos if v not in destinos),
        "sumideros": sum(1 for v in g.nodos if v not in origenes),
    }


def grados(aristas):
    entrada = defaultdict(int)
    salida = defaultdict(int)
    for a, b, _ in aristas:
        salida[a] += 1
        entrada[b] += 1
    return dict(entrada), dict(salida)


p = perfil(grafo_s1)
for clave, valor in p.items():
    texto = f"{valor:.4f}" if isinstance(valor, float) else f"{valor:,}"
    print(f"  {clave:<18} {texto}")

entrada, salida = grados(grafo_s1.aristas)


def frecuencias(grado):
    conteo = {}
    for g in grado.values():
        if g > 0:
            conteo[g] = conteo.get(g, 0) + 1
    return sorted(conteo.items())


fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for g, etiqueta, color in [(entrada, "entrada", "steelblue"), (salida, "salida", "darkorange")]:
    xs, ys = zip(*frecuencias(g))
    ejes[0].loglog(xs, ys, "o", label=f"grado de {etiqueta}", color=color, alpha=0.8)
ejes[0].set_xlabel("Grado")
ejes[0].set_ylabel("Número de nodos")
ejes[0].set_title("Distribución de grados (log-log)")
ejes[0].legend()

montos = [v for _, _, v, _ in filas if v > 0]
ejes[1].hist([math.log10(v) for v in montos], bins=30, color="seagreen", edgecolor="white")
ejes[1].set_xlabel("log10(valor en unidad mínima del token)")
ejes[1].set_ylabel("Frecuencia")
ejes[1].set_title("Monto por transferencia")
plt.tight_layout()
plt.show()

# Exchanges presentes y línea base por grado
presentes = {a: info for a, info in exchanges.items() if a in grafo_s1.nodos}
total = {v: entrada.get(v, 0) + salida.get(v, 0) for v in grafo_s1.nodos}

print()
print(f"Exchanges etiquetados presentes: {len(presentes)} de {len(exchanges)}")
print()
print(f"{'nombre':<24}{'tipo':<6}{'entrada':>9}{'salida':>8}{'percentil':>11}")
for addr in sorted(presentes, key=lambda a: -total[a]):
    info = presentes[addr]
    pct = 100 * sum(1 for x in total.values() if x < total[addr]) / max(1, len(total) - 1)
    print(f"{info['nombre'][:22]:<24}{info['tipo']:<6}{entrada.get(addr,0):>9}{salida.get(addr,0):>8}{pct:>10.1f}%")


def ranking(puntajes, mayor_es_mejor=True):
    signo = -1.0 if mayor_es_mejor else 1.0
    return sorted(puntajes, key=lambda n: (signo * puntajes[n], n))


def evaluar(puntajes, relevantes, ks):
    presentes_rel = {n for n in relevantes if n in puntajes}
    orden = ranking(puntajes)
    filas_eval = []
    for k in ks:
        enc = sum(1 for n in orden[:k] if n in presentes_rel)
        filas_eval.append((
            k,
            enc,
            enc / k if k else 0.0,
            enc / len(presentes_rel) if presentes_rel else 0.0,
        ))
    return filas_eval


if presentes:
    ks = [k for k in (10, 20, 50, 100) if k <= len(grafo_s1.nodos)]
    print()
    print("Línea base (precisión@k / recall@k) ordenando por grado:")
    print(f"{'puntaje':<16}" + "".join(f"{'k=' + str(k):>14}" for k in ks))
    for nombre, puntaje in [("grado total", total), ("grado entrada", entrada), ("grado salida", salida)]:
        filas_eval = evaluar(puntaje, presentes, ks)
        print(f"{nombre:<16}" + "".join(f"{f[2]:>7.2f} / {f[3]:<4.2f}" for f in filas_eval))
else:
    print("SoftChainCoin no tiene exchanges etiquetados en esta muestra.")

## Tarea 1 — Bellman-Ford: camino de costo mínimo

**Responsable:** Pablo Álvarez

El costo de un camino es la suma de los pesos de sus aristas. Bellman-Ford
relaja todas las aristas `|V|-1` veces y detecta ciclos negativos.

La implementación está en la celda siguiente y se prueba con la muestra.

In [ ]:
"""
bellman_ford.py
===============
Tarea 1 — Caminos de costo mínimo.

Problema: encontrar la ruta más "barata" entre dos direcciones del grafo,
donde el costo de una ruta es la suma de los pesos de sus aristas.

Regla del proyecto: NO se puede usar Dijkstra. Por eso usamos Bellman-Ford,
que además es capaz de detectar ciclos de peso negativo.

Idea del algoritmo:
    1. Empezamos con distancia 0 en el origen e "infinito" en el resto.
    2. Repetimos |V|-1 veces: por cada arista (u -> v, w), si
       dist[u] + w < dist[v], entonces actualizamos dist[v] = dist[u] + w.
       (a esto se le llama "relajar" una arista).
    3. Al terminar, dist[v] es la distancia mínima desde el origen a v.

Complejidad: O(|V| * |E|).
"""

from typing import Dict, List, Optional, Tuple

INF = float("inf")


def construir_adyacencia(aristas: List[Tuple[str, str, float]]) -> Dict[str, List[Tuple[str, float]]]:
    """Convierte la lista compartida de aristas en una lista de adyacencia.

    La lista de aristas viene de la carga única del dataset. Cada módulo
    construye su propia estructura, así no depende de la de otro.
    """
    from collections import defaultdict

    adj: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    for origen, destino, peso in aristas:
        adj[origen].append((destino, float(peso)))
    return dict(adj)


def bellman_ford(
    adj: Dict[str, List[Tuple[str, float]]],
    origen: str,
) -> Tuple[Dict[str, float], Dict[str, Optional[str]]]:
    """Calcula las distancias mínimas desde `origen` a todos los nodos.

    Parámetros:
        adj: lista de adyacencia {nodo: [(vecino, peso), ...]}.
        origen: nodo de partida.

    Retorna:
        (distancias, predecesores):
            - distancias[v] = costo mínimo desde `origen` hasta v
              (INF si v es inalcanzable).
            - predecesores[v] = nodo anterior a v en el camino mínimo
              (None para el origen).
    """
    # 1. Reunimos todos los nodos del grafo (incluye los que no tienen
    #    aristas salientes pero sí entrantes).
    nodos = set(adj.keys())
    for vecinos in adj.values():
        for dst, _ in vecinos:
            nodos.add(dst)

    # 2. Inicializamos distancias y predecesores.
    dist = {v: INF for v in nodos}
    dist[origen] = 0.0
    prev = {v: None for v in nodos}

    # 3. Relajamos todas las aristas |V|-1 veces.
    for _ in range(len(nodos) - 1):
        cambio = False
        for u in adj:
            for v, w in adj[u]:
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    prev[v] = u
                    cambio = True
        if not cambio:
            # Ya no mejora nada: terminamos antes de tiempo.
            break

    # 4. (Opcional) Una pasada extra para detectar ciclos de peso negativo.
    #    Con pesos positivos nunca ocurre, pero es una ventaja de Bellman-Ford.
    for u in adj:
        for v, w in adj[u]:
            if dist[u] + w < dist[v]:
                raise ValueError("Se detectó un ciclo de peso negativo.")

    return dist, prev


def reconstruir_camino(
    prev: Dict[str, Optional[str]],
    origen: str,
    destino: str,
) -> List[str]:
    """Reconstruye el camino mínimo desde `origen` hasta `destino`.

    Usa el diccionario de predecesores que devuelve `bellman_ford`.

    Retorna:
        Lista de nodos [origen, ..., destino], o [] si no hay camino.
    """
    camino: List[str] = []
    actual: Optional[str] = destino

    # Retrocedemos desde el destino hasta el origen siguiendo `prev`.
    while actual is not None:
        camino.append(actual)
        actual = prev.get(actual)

    camino.reverse()
    return camino if camino and camino[0] == origen else []


def bellman_ford_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    origen: str,
) -> Tuple[Dict[str, float], Dict[str, Optional[str]]]:
    """Igual que `bellman_ford`, pero recibe la lista compartida de aristas."""
    return bellman_ford(construir_adyacencia(aristas), origen)


# ============ DEMO TAREA 1 ============
from collections import Counter

salidas = Counter(origen for origen, _, _ in grafo.aristas)
etiquetados = sorted(
    (n for n in salidas if n in grafo.etiquetas),
    key=salidas.get, reverse=True,
)
origen = etiquetados[0] if etiquetados else max(salidas, key=salidas.get)

distancias, previos = bellman_ford_desde_aristas(grafo.aristas, origen)
alcanzables = sorted(
    (d, nodo) for nodo, d in distancias.items()
    if d != float("inf") and nodo != origen
)
inalcanzables = sum(1 for d in distancias.values() if d == float("inf"))

print(f"Origen: {grafo.nombre(origen)}")
print(f"Nodos alcanzables: {len(alcanzables):,}")
print(f"Nodos inalcanzables: {inalcanzables:,}\n")
print("Los 10 destinos más baratos:")
for d, destino in alcanzables[:10]:
    camino = reconstruir_camino(previos, origen, destino)
    nombres = " -> ".join(grafo.nombre(n) for n in camino)
    print(f"  {grafo.nombre(destino):<16} costo={d:>14,.2f}  ({nombres})")

## Tarea 2 — Centralidad y PageRank

**Responsable:** Jhon Arciniegas

- **Grado**: número de conexiones de un nodo.
- **PageRank**: importancia según quién apunta a quién.

PageRank se calcula con iteración de potencias y factor de amortiguación
`d = 0.85`.

La gráfica del top 5 y las validaciones de esta sección fueron aportadas por
Jhon Arciniegas (T2).

In [ ]:
"""
centralidad.py
==============
Tarea 2 — Centralidad e influencia.

Queremos responder: ¿qué nodos son los más importantes en la red?

Calculamos dos medidas:

    1. Centralidad de grado: cuántas conexiones tiene un nodo.
       C(v) = grado(v) / (|V| - 1)

    2. PageRank: la importancia de un nodo depende de la importancia de
       quienes le apuntan (es la misma idea que usa el buscador de Google).
       PR(v) = (1-d)/N + d * suma_{u -> v} ( PR(u) / grado_salida(u) )

       donde d es el factor de amortiguación (0.85 por defecto).
"""

from typing import Dict, List, Optional, Set, Tuple


def centralidad_grado(
    adj: Dict[str, List[Tuple[str, float]]],
    nodos: Set[str],
) -> Dict[str, float]:
    """Centralidad de grado normalizada: C(v) = grado(v) / (|V| - 1).

    En un grafo dirigido contamos grado de salida + grado de entrada.

    Parámetros:
        adj: lista de adyacencia.
        nodos: conjunto de todos los vértices.

    Retorna:
        {nodo: centralidad} con valores entre 0 y 1.
    """
    n = len(nodos)
    if n <= 1:
        return {v: 0.0 for v in nodos}

    grados = {v: 0 for v in nodos}
    for u, vecinos in adj.items():
        grados[u] += len(vecinos)   # grado de salida
        for v, _ in vecinos:
            grados[v] += 1          # grado de entrada

    return {v: grados[v] / (n - 1) for v in nodos}


def pagerank(
    adj: Dict[str, List[Tuple[str, float]]],
    nodos: Set[str],
    amortiguacion: float = 0.85,
    max_iter: int = 100,
    tol: float = 1e-6,
) -> Dict[str, float]:
    """Calcula PageRank con el método de iteración de potencias.

    Parámetros:
        adj: lista de adyacencia.
        nodos: conjunto de todos los vértices.
        amortiguacion: factor de amortiguación d (0.85 por defecto).
        max_iter: máximo número de iteraciones.
        tol: tolerancia de convergencia.

    Retorna:
        {nodo: score} con scores que suman aproximadamente 1.
    """
    n = len(nodos)
    if n == 0:
        return {}

    lista = list(nodos)

    # Puntaje inicial: todos los nodos igual de importantes (1/N).
    pr = {v: 1.0 / n for v in lista}

    # Grado de salida de cada nodo.
    grado_salida = {v: len(adj.get(v, [])) for v in lista}

    # Lista de adyacencia INVERSA: quién apunta a cada nodo.
    # Esto hace el cálculo más directo y eficiente.
    entrantes = {v: [] for v in lista}
    for u, vecinos in adj.items():
        for v, _ in vecinos:
            if v in entrantes:
                entrantes[v].append(u)

    for _ in range(max_iter):
        base = (1.0 - amortiguacion) / n

        # Los nodos "sumidero" (sin aristas de salida) no reparten su
        # importancia a nadie. Para que los puntajes sigan sumando 1, su masa
        # se reparte por igual entre todos los nodos.
        masa_sumidero = sum(pr[v] for v in lista if grado_salida[v] == 0)

        nuevo = {}
        for v in lista:
            suma = 0.0
            for u in entrantes[v]:
                if grado_salida[u] > 0:
                    suma += pr[u] / grado_salida[u]
            nuevo[v] = base + amortiguacion * (suma + masa_sumidero / n)

        # Medimos cuánto cambió el vector para saber si ya convergió.
        delta = sum(abs(nuevo[v] - pr[v]) for v in lista)
        pr = nuevo
        if delta < tol:
            break

    return pr


def top_n(
    puntajes: Dict[str, float],
    n: int = 10,
    etiquetas: Optional[Dict[str, Dict[str, str]]] = None,
) -> List[Tuple[str, float, str]]:
    """Devuelve los `n` nodos con mayor puntaje, ordenados de mayor a menor.

    Retorna:
        Lista de tuplas (dirección, puntaje, nombre) para mostrarlos bonito.
    """
    ordenados = sorted(puntajes.items(), key=lambda par: par[1], reverse=True)[:n]

    resultado = []
    for addr, score in ordenados:
        nombre = addr
        if etiquetas and addr in etiquetas:
            nombre = etiquetas[addr].get("nombre", addr)
        resultado.append((addr, round(score, 6), nombre))
    return resultado


def construir_adyacencia(aristas: List[Tuple[str, str, float]]) -> Dict[str, List[Tuple[str, float]]]:
    """Convierte la lista compartida de aristas en lista de adyacencia."""
    adj: Dict[str, List[Tuple[str, float]]] = {}
    for origen, destino, peso in aristas:
        adj.setdefault(origen, []).append((destino, float(peso)))
    return adj


def nodos_de_aristas(aristas: List[Tuple[str, str, float]]) -> Set[str]:
    """Devuelve todos los nodos que aparecen en la lista de aristas."""
    nodos: Set[str] = set()
    for origen, destino, _ in aristas:
        nodos.add(origen)
        nodos.add(destino)
    return nodos


def centralidad_grado_desde_aristas(
    aristas: List[Tuple[str, str, float]],
) -> Dict[str, float]:
    """Centralidad de grado a partir de la lista compartida de aristas."""
    return centralidad_grado(construir_adyacencia(aristas), nodos_de_aristas(aristas))


def pagerank_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    **opciones,
) -> Dict[str, float]:
    """PageRank a partir de la lista compartida de aristas."""
    return pagerank(
        construir_adyacencia(aristas),
        nodos_de_aristas(aristas),
        **opciones,
    )


# ============ DEMO TAREA 2 ============
import matplotlib.pyplot as plt

cent_grado = centralidad_grado_desde_aristas(grafo.aristas)
pr = pagerank_desde_aristas(grafo.aristas)

print("Top 10 por grado:")
for _, score, nombre in top_n(cent_grado, n=10, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<18} {score:.4f}")

print("\nTop 10 por PageRank:")
for _, score, nombre in top_n(pr, n=10, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<18} {score:.4f}")

# Gráfica y validaciones aportadas por Jhon Arciniegas (T2).
top_pr = top_n(pr, n=5, etiquetas=grafo.etiquetas)
nombres_pr = [nombre for _, _, nombre in top_pr]
scores_pr = [score for _, score, _ in top_pr]

plt.figure(figsize=(7, 4))
plt.barh(nombres_pr[::-1], scores_pr[::-1], color="steelblue")
plt.xlabel("PageRank")
plt.title("Top 5 direcciones por PageRank")
plt.tight_layout()
plt.show()

# Validaciones de centralidad (mismas que tests/test_centralidad.py).
assert centralidad_grado({"A": []}, {"A"})["A"] == 0.0
assert pagerank({}, set()) == {}
assert top_n({"A": 0.5, "B": 0.3}, n=2)[0][0] == "A"
print("Validaciones de centralidad: OK")

## Tarea 3 — Louvain: detección de comunidades

**Responsable:** Emerson Ballesteros

Se maximiza la modularidad:

$$Q = \sum_c \left[\frac{L_c}{m} - \left(\frac{K_c}{2m}\right)^2\right]$$

- `L_c`: peso interno de la comunidad.
- `K_c`: suma de grados de la comunidad.
- `m`: peso total del grafo.

`Q > 0.3` sugiere una estructura de comunidades clara.

In [ ]:
"""
comunidades.py
==============
Tarea 3 — Detección de comunidades.

Queremos agrupar los nodos en "comunidades": grupos de nodos que interactúan
mucho entre sí y poco con el resto de la red.

Medida de calidad: la modularidad Q (Newman-Girvan).

    Q = (1/2m) * suma_{i,j} [ A_ij - k_i*k_j/(2m) ] * delta(c_i, c_j)

    - A_ij: peso de la arista entre i y j.
    - k_i: grado (suma de pesos) del nodo i.
    - m: suma total de pesos del grafo.
    - delta(c_i, c_j): 1 si i y j están en la misma comunidad, 0 si no.

    Valores de Q por encima de ~0.3 indican una buena estructura de comunidades.

Algoritmo: usamos una versión sencilla de Louvain (su fase principal).

    - Cada nodo empieza en su propia comunidad.
    - Repetimos: para cada nodo, probamos moverlo a la comunidad de cada
      vecino y nos quedamos con el movimiento que más aumente Q.
    - Paramos cuando ningún movimiento mejore Q.

    (La versión "de producción" de Louvain usa una fórmula incremental para
    calcular el cambio de Q sin recalcularla toda; aquí la recalculamos cada
    vez para que el código sea fácil de leer. El grafo de ejemplo es pequeño.)
"""

import random
from collections import defaultdict
from typing import Dict, List, Set, Tuple


def simetrizar(adj: Dict[str, List[Tuple[str, float]]]) -> Dict[str, List[Tuple[str, float]]]:
    """Convierte el grafo dirigido en no dirigido.

    Si existe la arista u -> v con peso w, creamos la arista no dirigida
    {u, v} con peso w. Si existen en ambos sentidos, sumamos los pesos.

    Para comunidades trabajamos con el grafo no dirigido subyacente.
    """
    pesos = defaultdict(float)
    for u, vecinos in adj.items():
        for v, w in vecinos:
            a, b = sorted((u, v))   # ordenamos para no duplicar (u, v) y (v, u)
            pesos[(a, b)] += w

    sim = defaultdict(list)
    for (a, b), w in pesos.items():
        sim[a].append((b, w))
        sim[b].append((a, w))
    return dict(sim)


def modularidad(
    adj_nd: Dict[str, List[Tuple[str, float]]],
    comunidades: Dict[str, int],
) -> float:
    """Calcula la modularidad Q de la partición actual.

    Fórmula para grafo NO dirigido y ponderado:

        Q = suma_c [ L_c / m - (K_c / (2m))^2 ]

    donde:
        - L_c = peso de las aristas internas de la comunidad c.
        - K_c = suma de grados (pesos) de los nodos de c.
        - m   = peso total del grafo.

    Es equivalente a la fórmula clásica
        Q = (1/2m) * suma_{i,j} [A_ij - k_i*k_j/(2m)] * delta(c_i, c_j)
    pero se calcula por comunidad para no recorrer todos los pares.

    Parámetros:
        adj_nd: lista de adyacencia del grafo NO dirigido.
        comunidades: {nodo: id_comunidad}.

    Retorna:
        Q, un número entre -0.5 y 1.0.
    """
    grados = {v: sum(w for _, w in adj_nd.get(v, [])) for v in adj_nd}
    m = sum(grados.values()) / 2.0
    if m == 0:
        return 0.0

    # K_c: suma de grados de los nodos de cada comunidad.
    kc: Dict[int, float] = defaultdict(float)
    for v, k in grados.items():
        kc[comunidades[v]] += k

    # L_c: peso de las aristas internas. Cada arista no dirigida se cuenta
    # una sola vez, por eso usamos u < v (o el auto-loop una vez).
    lc: Dict[int, float] = defaultdict(float)
    for u, vecinos in adj_nd.items():
        for v, w in vecinos:
            if comunidades[u] != comunidades[v]:
                continue
            if u < v:
                lc[comunidades[u]] += w
            elif u == v:
                lc[comunidades[u]] += w / 2.0

    q = 0.0
    for c, k in kc.items():
        q += lc.get(c, 0.0) / m - (k / (2.0 * m)) ** 2
    return q


def louvain(
    adj_nd: Dict[str, List[Tuple[str, float]]],
    semilla: int = 42,
    max_pasadas: int = 20,
) -> Dict[str, int]:
    """Detecta comunidades maximizando la modularidad (Louvain, fase principal).

    Parámetros:
        adj_nd: lista de adyacencia del grafo NO dirigido.
        semilla: semilla para que el resultado sea reproducible.
        max_pasadas: máximo número de pasadas completas.

    Retorna:
        {nodo: id_comunidad}.
    """
    random.seed(semilla)
    nodos = list(adj_nd.keys())
    comunidades = {v: i for i, v in enumerate(nodos)}  # cada nodo en su comunidad

    for _ in range(max_pasadas):
        cambio = False
        random.shuffle(nodos)   # orden aleatorio para no sesgar el resultado

        for v in nodos:
            actual = comunidades[v]
            mejor = actual
            mejor_q = modularidad(adj_nd, comunidades)

            # Probamos mover v a la comunidad de cada vecino.
            for u, _ in adj_nd.get(v, []):
                c = comunidades[u]
                if c == actual:
                    continue

                comunidades[v] = c
                q = modularidad(adj_nd, comunidades)
                if q > mejor_q:
                    mejor_q = q
                    mejor = c
                comunidades[v] = actual   # deshacemos el cambio

            if mejor != actual:
                comunidades[v] = mejor
                cambio = True

        if not cambio:
            break   # ya nadie se mueve: convergió

    return comunidades


def resumen_comunidades(
    comunidades: Dict[str, int],
    etiquetas: Dict[str, Dict[str, str]] | None = None,
) -> Dict[int, List[str]]:
    """Agrupa los nodos por comunidad y devuelve sus nombres legibles.

    Parámetros:
        comunidades: {nodo: id_comunidad}.
        etiquetas: (opcional) {dirección: {"nombre": ...}}.

    Retorna:
        {id_comunidad: [nombres...]}.
    """
    grupos = defaultdict(list)
    for nodo, c in comunidades.items():
        nombre = nodo
        if etiquetas and nodo in etiquetas:
            nombre = etiquetas[nodo].get("nombre", nodo)
        grupos[c].append(nombre)
    return dict(grupos)


def desde_aristas(
    aristas: List[Tuple[str, str, float]],
) -> Dict[str, List[Tuple[str, float]]]:
    """Construye el grafo NO dirigido a partir de la lista compartida de aristas.

    Si hay transferencias en ambos sentidos, sus pesos se suman.
    """
    adj: Dict[str, List[Tuple[str, float]]] = {}
    for origen, destino, peso in aristas:
        adj.setdefault(origen, []).append((destino, float(peso)))
    return simetrizar(adj)


def louvain_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    semilla: int = 42,
    max_pasadas: int = 20,
) -> Dict[str, int]:
    """Louvain a partir de la lista compartida de aristas."""
    return louvain(desde_aristas(aristas), semilla=semilla, max_pasadas=max_pasadas)


# ============ DEMO TAREA 3 ============
adj_no_dirigido = desde_aristas(grafo.aristas)
comunidades = louvain_desde_aristas(grafo.aristas, semilla=42)
Q = modularidad(adj_no_dirigido, comunidades)

print(f"Modularidad Q = {Q:.3f}")
print(f"Número de comunidades = {len(set(comunidades.values()))}\n")

grupos = resumen_comunidades(comunidades, grafo.etiquetas)
for i, (c, nombres) in enumerate(
    sorted(grupos.items(), key=lambda kv: -len(kv[1]))[:10], 1
):
    muestra = ", ".join(sorted(nombres)[:5])
    if len(nombres) > 5:
        muestra += ", ..."
    print(f"Comunidad {i}: {len(nombres):>4} nodos | {muestra}")

# Estabilidad con varias semillas
print("\nEstabilidad con 5 semillas:")
valores_q = []
for s in [1, 7, 42, 123, 2024]:
    c = louvain(adj_no_dirigido, semilla=s)
    q = modularidad(adj_no_dirigido, c)
    valores_q.append(q)
    print(f"  semilla {s:>4}: Q={q:.3f}, comunidades={len(set(c.values()))}")

plt.figure(figsize=(6, 4))
plt.bar([str(s) for s in [1, 7, 42, 123, 2024]], valores_q, color="seagreen")
plt.xlabel("Semilla")
plt.ylabel("Modularidad Q")
plt.title("Sensibilidad de Louvain a la semilla")
plt.ylim(0, 1)
plt.show()

## Tarea 4 — Kruskal: árbol de expansión mínima

**Responsabilidad:** grupal

Se ordenan las aristas por peso y se agregan solo si no forman un ciclo,
usando **Union-Find**. El resultado tiene `|V|-1` aristas.

In [ ]:
"""
arbol_expansion.py
==================
Tarea 4 — Árbol de expansión mínima (MST).

Problema: dado un grafo ponderado, encontrar el conjunto de aristas que
conecta TODOS los nodos con el MENOR costo total posible. Ese conjunto
siempre forma un árbol (no tiene ciclos) y tiene exactamente |V|-1 aristas.

Algoritmo elegido: Kruskal.

    1. Ordenar las aristas por peso de menor a mayor.
    2. Recorrerlas en orden y agregar cada arista SOLO si no forma un ciclo.
    3. Para detectar ciclos usamos Union-Find (conjuntos disjuntos).

Union-Find es una estructura de datos clásica que permite saber rápido si
dos nodos ya están conectados (están en el mismo conjunto) y unirlos.
"""

from typing import Dict, List, Set, Tuple


class UnionFind:
    """Conjuntos disjuntos con compresión de caminos y unión por rango."""

    def __init__(self, nodos: Set[str]) -> None:
        # Cada nodo apunta a su "padre". Al inicio, cada uno es su propio padre.
        self.padre: Dict[str, str] = {v: v for v in nodos}
        # El "rango" es una cota de la altura del árbol de cada conjunto.
        self.rango: Dict[str, int] = {v: 0 for v in nodos}

    def encontrar(self, x: str) -> str:
        """Devuelve el representante (raíz) del conjunto al que pertenece x."""
        # Compresión de caminos: hacemos que cada nodo apunte directo a la raíz,
        # así las siguientes búsquedas son más rápidas.
        if self.padre[x] != x:
            self.padre[x] = self.encontrar(self.padre[x])
        return self.padre[x]

    def unir(self, x: str, y: str) -> bool:
        """Une los conjuntos de x e y.

        Retorna:
            True si estaban en conjuntos distintos (y los unió),
            False si ya estaban conectados (agregar la arista crearía un ciclo).
        """
        rx = self.encontrar(x)
        ry = self.encontrar(y)
        if rx == ry:
            return False   # ya conectados: esta arista formaría un ciclo

        # Unión por rango: el árbol más "bajo" cuelga del más "alto".
        if self.rango[rx] < self.rango[ry]:
            rx, ry = ry, rx
        self.padre[ry] = rx
        if self.rango[rx] == self.rango[ry]:
            self.rango[rx] += 1
        return True


def aristas_no_dirigidas(aristas: List[Tuple[str, str, float]]) -> List[Tuple[str, str, float]]:
    """Convierte aristas dirigidas en no dirigidas.

    Si aparecen (a, b, w1) y (b, a, w2), las juntamos en una sola arista
    {a, b} con peso w1 + w2.
    """
    pesos: Dict[Tuple[str, str], float] = {}
    for a, b, w in aristas:
        clave = tuple(sorted((a, b)))
        pesos[clave] = pesos.get(clave, 0.0) + w
    return [(a, b, w) for (a, b), w in pesos.items()]


def kruskal(
    nodos: Set[str],
    aristas: List[Tuple[str, str, float]],
) -> Tuple[List[Tuple[str, str, float]], float]:
    """Construye el árbol de expansión mínima con el algoritmo de Kruskal.

    Parámetros:
        nodos: conjunto de todos los vértices.
        aristas: lista de aristas NO dirigidas (a, b, peso).

    Retorna:
        (mst, costo_total) donde mst es la lista de aristas del árbol.
    """
    # 1. Ordenar las aristas por peso, de menor a mayor.
    ordenadas = sorted(aristas, key=lambda e: e[2])

    # 2. Estructura Union-Find para detectar ciclos.
    uf = UnionFind(nodos)

    mst: List[Tuple[str, str, float]] = []
    costo = 0.0
    objetivo = len(nodos) - 1   # un árbol con |V| nodos tiene |V|-1 aristas

    # 3. Recorrer las aristas en orden.
    for a, b, w in ordenadas:
        if len(mst) == objetivo:
            break   # ya tenemos el árbol completo

        if uf.unir(a, b):
            mst.append((a, b, w))
            costo += w

    return mst, costo


def resumen_mst(mst: List[Tuple[str, str, float]]) -> Dict[str, float]:
    """Resumen simple del MST: número de aristas y costo total."""
    return {
        "aristas": len(mst),
        "costo_total": sum(w for _, _, w in mst),
    }


def kruskal_desde_aristas(
    aristas: List[Tuple[str, str, float]],
) -> Tuple[List[Tuple[str, str, float]], float]:
    """Kruskal a partir de la lista compartida de aristas.

    Convierte las aristas dirigidas en no dirigidas (sumando pesos si hay
    transferencias en ambos sentidos) y calcula el MST.
    """
    no_dirigidas = aristas_no_dirigidas(
        [(origen, destino, float(peso)) for origen, destino, peso in aristas]
    )
    nodos: Set[str] = set()
    for origen, destino, _ in no_dirigidas:
        nodos.add(origen)
        nodos.add(destino)
    return kruskal(nodos, no_dirigidas)


# ============ DEMO TAREA 4 ============
mst, costo = kruskal_desde_aristas(grafo.aristas)
print(f"MST: {len(mst)} aristas, costo total = {costo:,.2f}\n")
print("Primeras 20 aristas del MST:")
for a, b, w in sorted(mst, key=lambda e: e[2])[:20]:
    print(f"  {grafo.nombre(a):<18} -- {grafo.nombre(b):<18} {w:,.2f}")

## 3. Análisis del dataset completo (opcional)

Si el Parquet completo está en Drive o en local, esta sección calcula
estadísticas sobre las **38.901.039 transferencias** con DuckDB, sin cargarlas
enteras en RAM.

Si no está, la celda se omite y el resto del notebook sigue funcionando.

In [ ]:
# ============ ANÁLISIS DEL DATASET COMPLETO ============
import os
import time

RUTA_PARQUET = "/content/transferencias.parquet"
RUTA_DRIVE = "/content/drive/MyDrive/ED2/transferencias.parquet"
RUTA_LOCAL = "data/procesado/transferencias.parquet"

# En Colab, descomenta si el Parquet está en Drive:
# from google.colab import drive
# drive.mount("/content/drive")

for _ruta in (RUTA_PARQUET, RUTA_DRIVE, RUTA_LOCAL):
    if os.path.isfile(_ruta):
        RUTA_PARQUET = _ruta
        break

if not os.path.isfile(RUTA_PARQUET):
    print("Sección omitida: no encontré el Parquet completo.")
    print("Súbelo a Drive como MyDrive/ED2/transferencias.parquet.")
else:
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow"], check=True)
        import duckdb

    con = duckdb.connect()
    con.execute("SET memory_limit='2GB'")
    con.execute("SET threads=2")
    con.execute("SET preserve_insertion_order=false")

    t0 = time.perf_counter()
    r = con.execute(f"""
        SELECT count(*) filas,
               count(DISTINCT token_address) tokens,
               min(block_number) bmin,
               max(block_number) bmax
        FROM read_parquet('{RUTA_PARQUET}')
    """).fetchone()
    print(f"Transferencias : {r[0]:,}")
    print(f"Tokens         : {r[1]}")
    print(f"Bloques        : {r[2]:,} .. {r[3]:,}")
    print(f"Tiempo         : {time.perf_counter()-t0:.1f} s")

    print("\nTop 10 tokens por transferencias:")
    for tok, n in con.execute(f"""
        SELECT token_address, count(*) n
        FROM read_parquet('{RUTA_PARQUET}')
        GROUP BY 1 ORDER BY n DESC LIMIT 10
    """).fetchall():
        print(f"  {tok}  {n:>10,}")

    print("\nActividad mensual aproximada:")
    for mes, n in con.execute(f"""
        SELECT strftime(to_timestamp(epoch(TIMESTAMP '2018-10-08') + (block_number-6525201)*13.5), '%Y-%m') mes,
               count(*) n
        FROM read_parquet('{RUTA_PARQUET}')
        GROUP BY mes ORDER BY mes
    """).fetchall():
        print(f"  {mes}: {n:>10,}")

### 3.1 Benchmark de escalabilidad

Extraemos subgrafos conectados reales del Parquet y medimos cuánto tarda cada
algoritmo. Si uno se pasa de 15 segundos, se marca como *timeout*.

Esto muestra, con datos reales, por qué Bellman-Ford y Louvain no pueden
ejecutarse sobre las 38,9 M de aristas en Python puro.

In [ ]:
# ============ BENCHMARK DE ESCALABILIDAD ============
if os.path.isfile(RUTA_PARQUET):
    import signal
    from collections import defaultdict, deque

    def subgrafo_conectado(aristas, max_aristas):
        ady = defaultdict(list)
        for i, (a, b, w) in enumerate(aristas):
            ady[a].append((b, w, i))
            ady[b].append((a, w, i))
        if not ady:
            return []
        inicio = max(ady, key=lambda n: len(ady[n]))
        visitados = {inicio}
        cola = deque([inicio])
        elegidas = set()
        while cola and len(elegidas) < max_aristas:
            u = cola.popleft()
            for v, w, i in ady[u]:
                if i in elegidas:
                    continue
                elegidas.add(i)
                if len(elegidas) >= max_aristas:
                    break
                if v not in visitados:
                    visitados.add(v)
                    cola.append(v)
        return [aristas[i] for i in sorted(elegidas)]

    def construir(aristas):
        adj = defaultdict(list)
        nodos = set()
        lista = []
        for a, b, w in aristas:
            adj[a].append((b, float(w)))
            nodos.add(a)
            nodos.add(b)
            lista.append((a, b, float(w)))
        return dict(adj), nodos, lista

    class _Timeout(Exception):
        pass

    def _handler(signum, frame):
        raise _Timeout()

    def medir(func, segundos):
        if not hasattr(signal, "SIGALRM"):
            t = time.perf_counter()
            func()
            return time.perf_counter() - t
        anterior = signal.signal(signal.SIGALRM, _handler)
        signal.setitimer(signal.ITIMER_REAL, segundos)
        try:
            t = time.perf_counter()
            func()
            return time.perf_counter() - t
        except _Timeout:
            return None
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, anterior)

    base = con.execute(f"""
        SELECT from_address, to_address, value
        FROM read_parquet('{RUTA_PARQUET}')
        USING SAMPLE reservoir(100000 ROWS) REPEATABLE (42)
    """).fetchall()

    tamanos = [1000, 5000, 10000, 50000, 100000]
    print(f"{'Algoritmo':<14}{'Aristas':>10}{'Nodos':>10}{'Tiempo (s)':>14}")
    for tam in tamanos:
        sub = subgrafo_conectado(base, tam)
        adj, nodos, aristas = construir(sub)
        if not nodos:
            continue
        origen = max(adj, key=lambda u: len(adj[u]))
        tareas = {
            "Bellman-Ford": lambda: bellman_ford(adj, origen),
            "PageRank": lambda: pagerank(adj, nodos),
            "Louvain": lambda: louvain(simetrizar(adj)),
            "Kruskal": lambda: kruskal(nodos, aristas_no_dirigidas(aristas)),
        }
        for nombre, func in tareas.items():
            t = medir(func, 15)
            marca = "timeout" if t is None else f"{t:.4f}"
            print(f"{nombre:<14}{len(aristas):>10,}{len(nodos):>10,}{marca:>14}")

## Conclusiones

1. **Un mismo dataset admite varias preguntas.** Bellman-Ford responde
   caminos, PageRank responde importancia, Louvain responde grupos y Kruskal
   responde conectividad mínima.

2. **Las estructuras de datos importan.** La lista de adyacencia, Union-Find y
   los diccionarios hacen que los algoritmos sean simples y legibles.

3. **La escala importa.** El análisis completo usa 38,9 M de transferencias con
   DuckDB; los algoritmos en Python puro se ejecutan sobre una muestra
   conectada de USDC.

4. **La muestra no reemplaza al dataset.** El Parquet conserva las 38.901.039
   filas; la muestra es solo para que los algoritmos terminen en Colab.

5. **Todo es reproducible.** La muestra tiene semilla 42 y el pipeline está
   documentado en el repositorio.